# M4 · XLM-RoBERTa (`xlm-roberta-base`)

Fourth model of the hierarchy of increasing multilingual capacity used in the paper.

| Model | Representation | Notebook |
|---|---|---|
| M1 | Bag-of-words + feed-forward network | `M1_bow.ipynb` |
| M2 | BiLSTM + frozen FastText vectors | `M2_bilstm_fasttext.ipynb` |
| M3 | mBERT (`bert-base-multilingual-cased`), full fine-tuning | `M3_mbert.ipynb` |
| M4 | XLM-RoBERTa (`xlm-roberta-base`), full fine-tuning | `M4_xlmr.ipynb` |

One fine-tuning run per source corpus and seed (42, 43, 44); the same model is evaluated on the test
partition of all six corpora, so diagonal and transfer cells come from the same run. Cross-lingual
cells are zero-shot (no translation). The predicted probabilities of every cell are saved to `.npz`.

XLM-RoBERTa sometimes collapses to the majority class during fine-tuning. A run whose validation
macro-F1 does not exceed the source trivial floor by more than 0.03 is discarded and repeated with
learning rate 1e-5, 20% warm-up and up to 8 epochs (Section 9). The rule was not triggered in the
run reported in the paper.

| Code | Corpus | Language | Platform | n |
|---|---|---|---|---|
| PT1 | HateBR (Vargas et al., 2022) | Portuguese | Instagram, political accounts | 7,000 |
| PT2 | OffComBR-3 (de Pelle & Moreira, 2017) | Portuguese | news portal comments | 1,033 |
| PT3 | ToLD-Br (Leite et al., 2020), stratified subsample | Portuguese | Twitter, general | 7,000 |
| EN | Davidson et al. (2017) | English | Twitter, general | 24,783 |
| ID | Alfina et al. (2017) | Indonesian | Twitter, political | 713 |
| KZ | Bolatbek et al. (2025) | Kazakh | YouTube, Telegram, VK | 10,190 |

**Inputs.** `protocolo_pibic.py` and the `splits/` folder in `BASE`, and the six corpus files in `BASE/dados/`
(see `data/corpora/README.md`). On Colab, `BASE` is `MyDrive/pibic`; elsewhere it is `./pibic`.

The construct-validity ablation on the English label, near the end of the notebook, was run later on
the same partitions and seeds. Its predictions are in the `*_20260819_*` folders
of `results/predictions/`.

## 1. Environment, Drive and dependencies

In [ ]:
# Do NOT upgrade torch on Colab; the preinstalled build matches the session GPU.
!pip install -q "transformers>=4.40" sentencepiece

import os, sys, json, gc, math, random, re, time, hashlib, unicodedata
from datetime import datetime
from contextlib import contextmanager
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# ── Google Drive (weights, results and diagnostics) ─────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/pibic'
except Exception:
    BASE = os.path.abspath('./pibic')       # local run
    print('Google Drive not available — using a local directory.')

DATA_DIR = os.path.join(BASE, 'dados')
OUT_DIR  = os.path.join(BASE, 'resultados')
SPLIT_DIR= os.path.join(BASE, 'splits')
HF_HOME  = os.path.join(BASE, 'hf_cache')
for d in (DATA_DIR, OUT_DIR, SPLIT_DIR, HF_HOME):
    os.makedirs(d, exist_ok=True)

# Hugging Face cache on Drive, so the checkpoint is not downloaded every session.
os.environ['HF_HOME'] = HF_HOME
os.environ['TRANSFORMERS_CACHE'] = HF_HOME
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Protocol shared by the four models: the ToLD-Br loader lives there so that
# M1, M2, M3 and M4 build EXACTLY the same PT3 corpus (same binarisation,
# same subsampling, same seed).
sys.path.insert(0, BASE)
import protocolo_pibic as PP

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Torch {torch.__version__} | device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: no GPU. Fine-tuning mBERT on CPU is not practical.')
print(f'BASE = {BASE}')

## 2. Experimental configuration

Every hyperparameter is declared here and saved with the results.

In [ ]:
CFG = {
    # ── Model ─────────────────────────────────────────────────────────────────
    'modelo_id'      : 'M4',
    'modelo_nome'    : 'XLM-RoBERTa (xlm-roberta-base)',
    'checkpoint'     : 'xlm-roberta-base',
    'max_len'        : 192,      # 99th percentile of KZ length = 192 subwords (mBERT tokenizer); see Section 7.
                                 # With dynamic padding, the extra cost falls only
                                 # on the few long examples.
    'marcadores'     : ['[URL]', '[USER]'],   # registered as special tokens

    # ── Optimisation (stability recipe of Mosbach et al., 2021) ─────────────────
    'lr'             : 2e-5,
    'weight_decay'   : 0.01,     # not applied to bias and LayerNorm
    'warmup_ratio'   : 0.10,
    'max_grad_norm'  : 1.0,
    'batch_size'     : 16,
    'batch_size_eval': 64,
    'epocas_max'     : 5,
    'paciencia'      : 2,        # early stopping on source validation macro-F1
    'label_smoothing': 0.0,
    'usar_amp'       : True,     # mixed precision (fp16)

    # ── Collapse recovery (XLM-R only) ────────────────────────────────────────
    # Collapse = validation macro-F1 stuck at the source trivial floor, i.e. the
    # model predicts a single class. Such a run is discarded and repeated.
    'margem_colapso' : 0.03,     # above the source trivial floor
    'lr_recuperacao' : 1e-5,
    'warmup_recuperacao': 0.20,
    'epocas_recuperacao': 8,

    # ── Protocol ──────────────────────────────────────────────────────────────
    'sementes'       : [42, 43, 44],
    'test_size'      : 0.20,     # test partition, stratified
    'val_size'       : 0.10,     # validation drawn from TRAIN (never from test)
    'ponderar_classes': True,    # class-weighted cross-entropy (weights from train only)

    # ── Threshold policy ──────────────────────────────────────────────────────
    # 'm1'  : calibrate the threshold on source validation only for in-corpus cells
    #         with >=100 validation examples; fixed 0.5 elsewhere (same as M1)
    # '05'  : fixed 0.5 everywhere
    'politica_limiar': 'm1',
    'min_val_calib'  : 100,

    # ── Label harmonisation ───────────────────────────────────────────────────
    # 'ampla'   : EN -> hate OR offensive = 1   (protocol default: 1 = problematic)
    # 'estrita' : EN -> hate only = 1, offensive dropped (sensitivity analysis)
    'harmonizacao_en': 'ampla',

    # ── Ablation ──────────────────────────────────────────────────────────────
    'ablacao_desacentuar_pt': True,   # see Section 15 (orthographic confounder PT1 x PT2)

    # ── PT3 · ToLD-Br ─────────────────────────────────────────────────────────
    # Simple majority (>=2 of 3 annotators) gives 19.3% positives, almost the
    # same as the 19.6% of OffComBR-3, so PT2 x PT3 is a domain contrast without a
    # base-rate confounder. Subsampling to 7,000 matches the size of HateBR and
    # removes corpus size as a competing explanation.
    'pt3_limiar_anotadores': 2,
    'pt3_n_max': 7000,          # None keeps all 21,000
}

# Groups of the experimental design.
# Group A is defined by LANGUAGE: every Portuguese-Portuguese pair is a
# within-language cross-domain cell (6 ordered cells over PT1/PT2/PT3).
LINGUA_SIGLA = {'PT1':'pt', 'PT2':'pt', 'PT3':'pt',
                'EN':'en', 'ID':'id', 'KZ':'kz'}

def grupo_cenario(src, tgt):
    if src == tgt:                              return 'diagonal'
    if LINGUA_SIGLA[src] == LINGUA_SIGLA[tgt]:  return 'A'   # same language, different domain
    if 'KZ' in (src, tgt):                      return 'C'   # maximum distance
    return 'B'                                               # cross-lingual, shorter distance

TAG = datetime.now().strftime('%Y%m%d_%H%M')
print(json.dumps(CFG, indent=2, ensure_ascii=False))

## 3. Utilities: reproducibility, timing and minimal text normalisation

In [ ]:
TEMPOS = defaultdict(float)

@contextmanager
def cronometro(rotulo, verboso=True):
    t0 = time.time()
    try:
        yield
    finally:
        dt = time.time() - t0
        TEMPOS[rotulo] += dt
        if verboso:
            print(f'  ⏱  {rotulo}: {dt/60:.2f} min')

def fixar_semente(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    # full determinism costs speed and is not needed: the variance across seeds is
    # what we want to measure. Only the starting point is fixed.
    torch.backends.cudnn.benchmark = True

def limpar_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

RE_ESCAPE_U = re.compile(r'\\u([0-9a-fA-F]{4})')

def decodificar_escapes(t):
    # The Indonesian file stores text with JSON-style escapes. Emoji are stored as
    # SURROGATE PAIRS (\\ud83d\\ude02 = one character). Decoding each escape alone
    # with chr() yields LONE surrogates, which are not valid UTF-8.
    # The fast tokenizer in transformers is written in Rust and requires valid
    # UTF-8: a lone surrogate makes the call fail with
    #   TypeError: TextEncodeInput must be Union[TextInputSequence, ...]
    # Re-encoding through UTF-16 below recombines the pairs into the original character.
    t = t.replace('\\/', '/')
    t = RE_ESCAPE_U.sub(lambda m: chr(int(m.group(1), 16)), t)
    try:
        t = t.encode('utf-16', 'surrogatepass').decode('utf-16')
    except UnicodeError:
        pass
    return t

def sanitizar(t):
    # Safety net: drop any remaining lone surrogate.
    t = str(t)
    try:
        t.encode('utf-8')
        return t
    except UnicodeEncodeError:
        return t.encode('utf-8', 'ignore').decode('utf-8')

RE_URL  = re.compile(r'https?://\S+|www\.\S+')
RE_USER = re.compile(r'@\w+')
RE_ESP  = re.compile(r'\s+')

def normalizar_minimo(t):
    # Defined here (and not in Section 8) so that the tokenisation diagnostic in
    # Section 7 measures fertility on the text the model actually receives.
    t = RE_URL.sub('[URL]', sanitizar(t))
    t = RE_USER.sub('[USER]', t)
    return RE_ESP.sub(' ', t).strip()

def sem_acento(s):
    return ''.join(c for c in unicodedata.normalize('NFKD', str(s))
                   if not unicodedata.combining(c))

# Normalised key for duplicate detection within and across corpora.
def chave_norm(s):
    s = sem_acento(str(s)).lower()
    s = re.sub(r'http\S+|www\.\S+', ' ', s)
    s = re.sub(r'[^a-z0-9\u0400-\u04FF ]', ' ', s)
    return ' '.join(s.split())

print('Utilities loaded.')

## 4. Loading and harmonising the corpora

All corpora are mapped to a binary label: **1 = problematic**, **0 = not problematic**. The mapping
merges categories that the original corpora keep apart:

- **EN (Davidson et al.):** `hate` and `offensive` both become 1, so the positive class is dominated by
  offensive language (5.8% of the corpus is hate speech in the strict sense).
- **KZ (Bolatbek et al.):** racism, cyberbullying, religious extremism and national extremism become 1.
- **PT1 (HateBR):** `label_final` (offensiveness). **PT2 (OffComBR-3):** `yes`/`no`.
- **PT3 (ToLD-Br):** positive if any of the six categories is marked by at least 2 of 3 annotators.
- **ID (Alfina et al.):** `HS`/`Non_HS`.

The matrix therefore measures transfer of problematic content, not of hate speech in the strict sense.

In [ ]:
def _ler_indonesio(caminho):
    bruto = open(caminho, encoding='utf-8', errors='ignore').read().replace('\r\n','\n').replace('\r','\n')
    linhas = [l for l in bruto.split('\n') if '\t' in l][1:]
    textos, rotulos = [], []
    for l in linhas:
        rot, txt = l.split('\t', 1)
        txt = decodificar_escapes(txt)
        textos.append(txt.strip().strip('"'))
        rotulos.append(1 if rot.strip() == 'HS' else 0)
    return textos, rotulos

def _ler_arff(caminho):
    bruto = open(caminho, encoding='utf-8', errors='ignore').read()
    dados = bruto.split('@data', 1)[1].strip().split('\n')
    textos, rotulos = [], []
    for l in dados:
        l = l.strip()
        if not l or ',' not in l: continue
        rot, txt = l.split(',', 1)
        rot = rot.strip().lower()
        if rot not in ('yes','no'): continue
        textos.append(txt.strip().strip("'").strip())
        rotulos.append(1 if rot == 'yes' else 0)
    return textos, rotulos

def carregar_corpora(data_dir=DATA_DIR):
    C = {}

    # ── PT1 · HateBR ──────────────────────────────────────────────────────────
    df = pd.read_csv(os.path.join(data_dir, 'HateBR.csv'))
    C['PT1'] = dict(textos=df['comentario'].astype(str).tolist(),
                    rotulos=df['label_final'].astype(int).tolist(),
                    lang='pt', codigo_iso='pt',
                    nome='HateBR', dominio='político / Instagram',
                    referencia='Vargas et al. (2022)')

    # ── PT2 · OffComBR-3 ──────────────────────────────────────────────────────
    t, y = _ler_arff(os.path.join(data_dir, 'OffComBR3.arff'))
    C['PT2'] = dict(textos=t, rotulos=y, lang='pt', codigo_iso='pt',
                    nome='OffComBR-3', dominio='comentários de portal de notícias',
                    referencia='Pelle & Moreira (2017)')

    # ── PT3 · ToLD-Br ─────────────────────────────────────────────────────────
    # Leite, Silva, Bontcheva & Scarton (2020), AACL-IJCNLP. License: CC BY-SA 4.0.
    # Download ToLD-BR.csv from https://github.com/joaoaleite/ToLD-Br into DATA_DIR.
    t, y, meta_pt3 = PP.carregar_toldbr(
        os.path.join(data_dir, 'ToLD-BR.csv'),
        limiar_anotadores=CFG['pt3_limiar_anotadores'],
        n_max=CFG['pt3_n_max'])
    C['PT3'] = dict(textos=t, rotulos=y, lang='pt', codigo_iso='pt',
                    nome='ToLD-Br', dominio='Twitter, geral',
                    referencia='Leite et al. (2020)', meta=meta_pt3)

    # ── EN · Davidson et al. ──────────────────────────────────────────────────
    df = pd.read_csv(os.path.join(data_dir, 'DataIngles.csv'))
    if CFG['harmonizacao_en'] == 'ampla':
        y = (df['class'] != 2).astype(int).tolist()      # hate|offensive = 1
        t = df['tweet'].astype(str).tolist()
    else:                                                 # 'estrita': drops offensive
        m = df['class'] != 1
        y = (df.loc[m,'class'] == 0).astype(int).tolist()
        t = df.loc[m,'tweet'].astype(str).tolist()
    C['EN'] = dict(textos=t, rotulos=y, lang='en', codigo_iso='en',
                   nome='Davidson et al.', dominio='Twitter, geral',
                   referencia='Davidson et al. (2017)')

    # ── ID · Alfina et al. (2017) ─────────────────────────────────────────────
    t, y = _ler_indonesio(os.path.join(data_dir, 'DataIndonesio.txt'))
    C['ID'] = dict(textos=t, rotulos=y, lang='id', codigo_iso='id',
                   nome='Alfina et al.', dominio='Twitter, político',
                   referencia='Alfina et al. (2017)')

    # ── KZ · Bolatbek et al. ──────────────────────────────────────────────────
    # Uses 'message' (raw text), NOT 'message_stemmed': stemming destroys the
    # morphology the subword tokenizer needs to segment Kazakh.
    df = pd.read_csv(os.path.join(data_dir, 'DataKazake.csv'))
    C['KZ'] = dict(textos=df['message'].astype(str).tolist(),
                   rotulos=(df['label'] != 0).astype(int).tolist(),
                   lang='kz', codigo_iso='kk',
                   nome='Bolatbek et al.', dominio='YouTube / Telegram / VK',
                   referencia='Bolatbek et al. (2025)')

    # Encoding clean-up applied to ALL corpora, with counts logged.
    C['_saneados'] = {}
    for k, c in C.items():
        if k.startswith('_'): continue
        limpos, n_alterados = [], 0
        for t in c['textos']:
            s = sanitizar(t)
            if s != str(t): n_alterados += 1
            limpos.append(s)
        c['textos'] = limpos
        C['_saneados'][k] = n_alterados

    for k, c in C.items():
        if k.startswith('_'): continue
        c['sigla'] = k
        c['n'] = len(c['textos'])
        c['pos'] = int(sum(c['rotulos']))
        c['prev_pos'] = c['pos'] / c['n']
        # Trivial floor: macro-F1 of the majority-class classifier
        p = max(c['prev_pos'], 1 - c['prev_pos'])
        c['baseline_majoritaria'] = 0.5 * (2*p / (1 + p))     # F1 of the majority class / 2
    return C

with cronometro('carregamento_dados'):
    CORPORA = carregar_corpora()
SANEADOS = CORPORA.pop('_saneados')
if any(SANEADOS.values()):
    print('Texts fixed by the encoding clean-up: '
          + ', '.join(f'{k}={v}' for k, v in SANEADOS.items() if v) + '\n')

print(f"{'Code':<5} {'Corpus':<18} {'Lang':<5} {'N':>7} {'%pos':>7} {'trivial macroF1':>16}")
print('-'*66)
for k, c in CORPORA.items():
    print(f"{k:<5} {c['nome']:<18} {c['lang']:<5} {c['n']:>7,} "
          f"{c['prev_pos']*100:>6.1f}% {c['baseline_majoritaria']:>16.3f}")

## 5. Corpus integrity diagnostics

Three checks, recorded in the output JSON: internal duplicates, overlap across corpora (removed from
the test partitions) and surface artefacts correlated with the label.

In [ ]:
DIAGNOSTICOS = {}

with cronometro('diagnostico_integridade'):
    # ── 5.1 Internal duplicates ───────────────────────────────────────────────
    dup_int = {}
    for k, c in CORPORA.items():
        chaves = [chave_norm(t) for t in c['textos']]
        c['_chaves'] = chaves
        n_dup = len(chaves) - len(set(chaves))
        # Duplicates with conflicting labels: a symptom of annotation noise
        m = defaultdict(set)
        for ch, y in zip(chaves, c['rotulos']): m[ch].add(y)
        conflito = sum(1 for v in m.values() if len(v) > 1)
        dup_int[k] = dict(duplicatas=n_dup, chaves_com_rotulo_conflitante=conflito)
    DIAGNOSTICOS['duplicatas_internas'] = dup_int

    # ── 5.2 Overlap across corpora ────────────────────────────────────────────
    sob = {}
    siglas = list(CORPORA)
    for i, a in enumerate(siglas):
        for b in siglas[i+1:]:
            inter = set(CORPORA[a]['_chaves']) & set(CORPORA[b]['_chaves'])
            inter.discard('')
            if inter: sob[f'{a}-{b}'] = len(inter)
    DIAGNOSTICOS['sobreposicao_entre_corpora'] = sob

print('── Internal duplicates ──')
for k, v in DIAGNOSTICOS['duplicatas_internas'].items():
    print(f"  {k:<4} duplicates={v['duplicatas']:>4}  conflicting label={v['chaves_com_rotulo_conflitante']:>3}")
print('\n── Overlap across corpora (normalised key) ──')
print('  ' + (str(DIAGNOSTICOS['sobreposicao_entre_corpora']) or 'nenhuma'))

### 5.3 Surface-shortcut probe: separability without content

A logistic regression that only sees length, upper case, punctuation, URLs and alphabet is trained to
separate the classes of each corpus.

- **Surface AUC:** 0.5 means no shortcut.
- **Spearman ρ between length and label:** two corpora with opposite signs and similar magnitude
  can produce transfer AUC below 0.5 between them.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from scipy.stats import spearmanr

def features_superficiais(textos):
    F = []
    for t in textos:
        t = str(t)
        n_car = max(1, len(t))
        alpha = max(1, sum(c.isalpha() for c in t))
        F.append([
            len(t.split()),                                    # words
            len(t),                                            # characters
            sum(c.isascii() and c.isalpha() for c in t)/alpha,  # share of Latin characters
            sum(c.isupper() for c in t)/n_car,                  # upper case
            sum(c in '!?' for c in t)/n_car,                    # emphatic punctuation
            int(bool(re.search(r'http|www\.', t))),             # URL present
        ])
    return np.array(F)

with cronometro('sondagem_atalhos'):
    linhas = []
    for k, c in CORPORA.items():
        X = features_superficiais(c['textos']); y = np.array(c['rotulos'])
        clf = make_pipeline(StandardScaler(),
                            LogisticRegression(max_iter=1000, class_weight='balanced'))
        auc = cross_val_score(clf, X, y, cv=5, scoring='roc_auc').mean()
        n_pal = np.array([len(str(t).split()) for t in c['textos']])
        rho = spearmanr(n_pal, y).statistic
        linhas.append(dict(corpus=k, lingua=c['lang'], prevalencia=c['prev_pos'],
                           auc_superficial=float(auc), rho_comprimento=float(rho)))

SOND = pd.DataFrame(linhas)
DIAGNOSTICOS['sondagem_atalhos'] = SOND.to_dict(orient='records')

print(f"{'Corpus':<7} {'Lang':<7} {'%pos':>7} {'surface AUC':>17} {'length rho':>12}")
print('-'*54)
for _, r in SOND.iterrows():
    flag = '  ⚠' if r['auc_superficial'] > 0.65 else ''
    print(f"{r['corpus']:<7} {r['lingua']:<7} {r['prevalencia']*100:>6.1f}% "
          f"{r['auc_superficial']:>17.3f} {r['rho_comprimento']:>+12.3f}{flag}")

print('\nSurface AUC close to 0.5 = no surface shortcut.')
print('The SIGN of rho matters: corpora with opposite signs produce transfer')
print('that is systematically INVERTED between them (AUC < 0.5); see the check below.')

print('\n── Pairs with opposite rho signs ──')
pares_invertidos = []
sig = list(SOND['corpus'])
for i_ in range(len(sig)):
    for j_ in range(i_+1, len(sig)):
        a, b = sig[i_], sig[j_]
        ra = SOND[SOND.corpus==a]['rho_comprimento'].iloc[0]
        rb = SOND[SOND.corpus==b]['rho_comprimento'].iloc[0]
        if ra * rb < 0 and min(abs(ra), abs(rb)) > 0.10:
            pares_invertidos.append((a, b, ra, rb))
            print(f'  {a} (rho={ra:+.3f})  x  {b} (rho={rb:+.3f})  '
                  f'→ expect transfer AUC < 0.5 between them')
if not pares_invertidos:
    print('  none')
DIAGNOSTICOS['pares_rho_oposto'] = [
    dict(a=a, b=b, rho_a=float(ra), rho_b=float(rb)) for a, b, ra, rb in pares_invertidos]

## 6. Fixed partitions shared by M1–M4

Test: 20%, stratified by label. Validation: 10% of the remaining data, drawn from train, used for
early stopping and threshold calibration. Internal duplicates and texts that also occur in another
corpus are removed from the test partitions only. The partitions are loaded from `splits/`; if the
output does not report a loaded partition for every corpus, check the path.

In [ ]:
from sklearn.model_selection import train_test_split

def obter_particoes(sigla, corpus, remover_duplicatas_teste=True):
    caminho = os.path.join(SPLIT_DIR, f'split_{sigla}.json')
    n = corpus['n']

    if os.path.exists(caminho):
        with open(caminho) as f: sp = json.load(f)
        if sp.get('n') == n:
            print(f'  {sigla}: partition loaded from {os.path.basename(caminho)}')
            return sp['treino'], sp['val'], sp['teste']
        print(f'  {sigla}: size changed ({sp.get("n")} -> {n}); regenerating partition.')

    idx = np.arange(n); y = np.array(corpus['rotulos'])
    tr_full, te = train_test_split(idx, test_size=CFG['test_size'],
                                   random_state=42, stratify=y)
    tr, va = train_test_split(tr_full, test_size=CFG['val_size'],
                              random_state=42, stratify=y[tr_full])

    if remover_duplicatas_teste:
        chaves = corpus['_chaves']
        vistas = set(chaves[i] for i in tr) | set(chaves[i] for i in va)
        externas = set()
        for outro, c2 in CORPORA.items():
            if outro != sigla: externas |= set(c2['_chaves'])
        limpo, desc_int, desc_ext, local = [], 0, 0, set()
        for i in te:
            ch = chaves[i]
            if ch in vistas or ch in local: desc_int += 1; continue
            if ch and ch in externas:       desc_ext += 1; continue
            local.add(ch); limpo.append(int(i))
        if desc_int or desc_ext:
            print(f'  {sigla}: test cleaned — {desc_int} internal duplicates, '
                  f'{desc_ext} overlaps with other corpora')
        te = np.array(limpo)

    sp = dict(n=n, treino=[int(i) for i in tr], val=[int(i) for i in va],
              teste=[int(i) for i in te], semente_split=42,
              gerado_em=datetime.now().isoformat())
    with open(caminho, 'w') as f: json.dump(sp, f)
    return sp['treino'], sp['val'], sp['teste']

with cronometro('particoes'):
    for k, c in CORPORA.items():
        c['idx_treino'], c['idx_val'], c['idx_teste'] = obter_particoes(k, c)

# Explicit check: the partitions MUST come from disk.
_faltando = [k for k in CORPORA
             if not os.path.exists(os.path.join(SPLIT_DIR, f'split_{k}.json'))]
if _faltando:
    print(f'\n⚠ WARNING: partitions generated now for {_faltando}. If M3 has already run,')
    print('  the path is wrong and the M3 × M4 comparison is compromised.')
    print('  Check SPLIT_DIR before continuing.')
else:
    print('\n✓ All partitions came from disk — the comparison with M3 is valid.')

print(f"\n{'Corpus':<6} {'train':>8} {'val':>7} {'test':>7} {'%pos train':>13} {'%pos test':>12}")
print('-'*58)
for k, c in CORPORA.items():
    y = np.array(c['rotulos'])
    print(f"{k:<6} {len(c['idx_treino']):>8,} {len(c['idx_val']):>7,} {len(c['idx_teste']):>7,} "
          f"{y[c['idx_treino']].mean()*100:>12.1f}% {y[c['idx_teste']].mean()*100:>11.1f}%")

## 7. Tokenizer: fertility and coverage by language

Fertility (subwords per word) and the rate of unknown tokens per corpus, on a sample of 3,000 texts
from each corpus. The maximum length of 192 subwords is the 99th percentile of the Kazakh corpus under
the mBERT tokenizer (147 under XLM-R). The `[URL]` and `[USER]` markers are registered as special tokens.

In [ ]:
from transformers import AutoTokenizer

with cronometro('carregar_tokenizador'):
    TOK = AutoTokenizer.from_pretrained(CFG['checkpoint'])

# The [URL] and [USER] markers are NOT in the tokenizer vocabulary: without
# explicit registration each splits into 4-5 noise subwords, which inflates
# fertility and wastes context, a large effect on tweet corpora (EN, ID).
# Registered as special tokens they become 1 token each; the new rows of the
# embedding matrix start random and are learned during fine-tuning
# (see construir_modelo, which calls resize_token_embeddings).
# In XLM-R the tokenizer is SentencePiece (unigram) and the unknown token is <unk>,
# not [UNK]. The diagnostic below uses TOK.unk_token, so it works for both.
print('Before registration:', TOK.tokenize(' '.join(CFG['marcadores'])))
n_novos = TOK.add_special_tokens({'additional_special_tokens': CFG['marcadores']})
print(f'After registration ({n_novos} new):',
      TOK.tokenize(' '.join(CFG['marcadores'])))
print(f'Vocabulary size: {len(TOK):,}\n')

def diagnostico_tokenizacao(textos, amostra=3000, semente=42, normalizar=True):
    # By default, measure on the NORMALISED text, which is what the model receives.
    # Measuring on raw text inflates the fertility of tweet corpora (raw URLs and
    # HTML entities).
    rng = np.random.RandomState(semente)
    sel = rng.choice(len(textos), size=min(amostra, len(textos)), replace=False)
    n_pal = n_sub = n_unk = 0; comprimentos = []
    for i in sel:
        t = normalizar_minimo(textos[i]) if normalizar else str(textos[i])
        palavras = t.split()
        n_pal += len(palavras)
        ids = TOK(t, add_special_tokens=True, truncation=True, max_length=512)['input_ids']
        toks = TOK.convert_ids_to_tokens(ids)
        n_sub += len(toks) - 2
        n_unk += sum(1 for x in toks if x == TOK.unk_token)
        comprimentos.append(len(ids))
    c = np.array(comprimentos)
    return dict(fertilidade=n_sub/max(1,n_pal), taxa_unk=n_unk/max(1,n_sub),
                p50=int(np.percentile(c,50)), p95=int(np.percentile(c,95)),
                p99=int(np.percentile(c,99)), max=int(c.max()),
                trunc_128=float((c>128).mean()))

with cronometro('diagnostico_tokenizacao'):
    DIAGNOSTICOS['tokenizacao'] = {k: diagnostico_tokenizacao(c['textos'])
                                   for k, c in CORPORA.items()}
    # Also stored for raw text, for comparison with the M3 table.
    DIAGNOSTICOS['tokenizacao_bruto'] = {
        k: diagnostico_tokenizacao(c['textos'], normalizar=False)
        for k, c in CORPORA.items()}

print(f"{'Corpus':<6} {'Lang':<7} {'fertil.':>8} {'[UNK]':>8} {'p50':>5} {'p95':>5} {'p99':>5} {'>128':>7}")
print('-'*62)
for k, d in DIAGNOSTICOS['tokenizacao'].items():
    print(f"{k:<6} {CORPORA[k]['lang']:<7} {d['fertilidade']:>8.2f} {d['taxa_unk']*100:>7.2f}% "
          f"{d['p50']:>5} {d['p95']:>5} {d['p99']:>5} {d['trunc_128']*100:>6.1f}%")

print('\n── Same measure on the RAW text (comparable with the M3 table) ──')
print(f"{'Corpus':<6} {'fertil.':>8} {'unknown':>15}")
print('-'*32)
for k, d in DIAGNOSTICOS['tokenizacao_bruto'].items():
    print(f"{k:<6} {d['fertilidade']:>8.2f} {d['taxa_unk']*100:>14.2f}%")
print('\nThe difference between the two tables is the effect of masking URLs and mentions.')
print('The normalised table describes the model input; the raw one is comparable with M3.')

p99_max = max(d['p99'] for d in DIAGNOSTICOS['tokenizacao'].values())
print(f'\nLargest p99 across corpora: {p99_max} subwords.')
print(f"max_len set to: {CFG['max_len']}.")
if p99_max > CFG['max_len']:
    print('→ Consider a larger max_len: truncating more than 1% of the examples in one corpus')
    print('  and not in another adds an asymmetry between cells of the matrix.')

## 8. Dataset and batching

Minimal normalisation only: URLs → `[URL]`, mentions → `[USER]`, whitespace collapsed. Case, accents,
emoji and punctuation are kept. Dynamic padding to the longest example of each batch.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import DataCollatorWithPadding

# normalizar_minimo() is defined in Section 3 so that the tokenisation
# diagnostic in Section 7 can use it.

class CorpusOdio(Dataset):
    def __init__(self, textos, rotulos, tok, max_len, desacentuar=False):
        self.textos = [normalizar_minimo(t) for t in textos]
        if desacentuar:
            self.textos = [sem_acento(t) for t in self.textos]
        self.rotulos = list(rotulos)
        self.tok, self.max_len = tok, max_len
    def __len__(self):
        return len(self.rotulos)
    def __getitem__(self, i):
        enc = self.tok(self.textos[i], truncation=True, max_length=self.max_len)
        enc['labels'] = int(self.rotulos[i])
        return enc

COLLATOR = DataCollatorWithPadding(tokenizer=TOK, padding='longest', return_tensors='pt')

def montar_dataset(sigla, particao, desacentuar=False):
    c = CORPORA[sigla]
    idx = {'treino': c['idx_treino'], 'val': c['idx_val'], 'teste': c['idx_teste']}[particao]
    return CorpusOdio([c['textos'][i] for i in idx],
                      [c['rotulos'][i] for i in idx],
                      TOK, CFG['max_len'], desacentuar=desacentuar)

def montar_loader(ds, treino=False):
    return DataLoader(ds,
                      batch_size=CFG['batch_size'] if treino else CFG['batch_size_eval'],
                      shuffle=treino, collate_fn=COLLATOR,
                      num_workers=2, pin_memory=(DEVICE.type=='cuda'),
                      drop_last=False)

print('Dataset and collator defined.')

## 9. Model, loss and fine-tuning loop

Two-logit classification head and class-weighted cross-entropy, with weights computed on the training
partition only (`w_c = n / (2 · n_c)`). Weight decay is not applied to bias and LayerNorm. Early
stopping on source validation macro-F1; the best weights are restored at the end.

In [ ]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import (f1_score, precision_recall_fscore_support,
                             accuracy_score, roc_auc_score, average_precision_score,
                             confusion_matrix, classification_report)

def pesos_classe(rotulos):
    y = np.asarray(rotulos); n = len(y)
    w = np.array([n / (2 * max(1, (y == c).sum())) for c in (0, 1)], dtype=np.float32)
    return torch.tensor(w, device=DEVICE)

def construir_modelo():
    m = AutoModelForSequenceClassification.from_pretrained(
        CFG['checkpoint'], num_labels=2)
    # Makes room for the markers registered in Section 7. Without it the new ids
    # would point outside the embedding matrix and the forward pass would fail.
    if len(TOK) != m.get_input_embeddings().weight.shape[0]:
        m.resize_token_embeddings(len(TOK))
    return m.to(DEVICE)

@torch.no_grad()
def inferir_probs(modelo, loader):
    modelo.eval()
    P, Y = [], []
    for lote in loader:
        y = lote.pop('labels')
        lote = {k: v.to(DEVICE, non_blocking=True) for k, v in lote.items()}
        with torch.autocast('cuda', dtype=torch.float16,
                            enabled=(CFG['usar_amp'] and DEVICE.type=='cuda')):
            logits = modelo(**lote).logits
        P.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy())
        Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)

def ajustar_fino(sigla_fonte, semente, desacentuar=False, verboso=True,
                 lr=None, warmup_ratio=None, epocas_max=None):
    lr = CFG['lr'] if lr is None else lr
    warmup_ratio = CFG['warmup_ratio'] if warmup_ratio is None else warmup_ratio
    epocas_max = CFG['epocas_max'] if epocas_max is None else epocas_max
    fixar_semente(semente)
    ds_tr = montar_dataset(sigla_fonte, 'treino', desacentuar)
    ds_va = montar_dataset(sigla_fonte, 'val',    desacentuar)
    ld_tr, ld_va = montar_loader(ds_tr, treino=True), montar_loader(ds_va)

    modelo = construir_modelo()
    pesos = pesos_classe(ds_tr.rotulos) if CFG['ponderar_classes'] else None
    criterio = nn.CrossEntropyLoss(weight=pesos, label_smoothing=CFG['label_smoothing'])

    sem_decay = ['bias', 'LayerNorm.weight']
    grupos = [
        {'params': [p for n_, p in modelo.named_parameters()
                    if not any(nd in n_ for nd in sem_decay)],
         'weight_decay': CFG['weight_decay']},
        {'params': [p for n_, p in modelo.named_parameters()
                    if any(nd in n_ for nd in sem_decay)],
         'weight_decay': 0.0},
    ]
    otim = torch.optim.AdamW(grupos, lr=lr, eps=1e-8)   # torch AdamW already applies bias correction
    total = len(ld_tr) * epocas_max
    agenda = get_linear_schedule_with_warmup(
        otim, int(warmup_ratio * total), total)
    _amp = CFG['usar_amp'] and DEVICE.type == 'cuda'
    try:                                   # torch >= 2.4
        escala = torch.amp.GradScaler('cuda', enabled=_amp)
    except (AttributeError, TypeError):     # older torch
        escala = torch.cuda.amp.GradScaler(enabled=_amp)

    melhor_f1, melhor_estado, melhor_ep, paciencia = -1.0, None, 0, 0
    historico = []

    for ep in range(1, epocas_max + 1):
        modelo.train(); soma = 0.0
        for lote in ld_tr:
            y = lote.pop('labels').to(DEVICE, non_blocking=True)
            lote = {k: v.to(DEVICE, non_blocking=True) for k, v in lote.items()}
            otim.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16,
                                enabled=(CFG['usar_amp'] and DEVICE.type=='cuda')):
                logits = modelo(**lote).logits
                perda = criterio(logits.float(), y)
            escala.scale(perda).backward()
            escala.unscale_(otim)
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), CFG['max_grad_norm'])
            escala.step(otim); escala.update(); agenda.step()
            soma += perda.item()

        p_va, y_va = inferir_probs(modelo, ld_va)
        f1_va = f1_score(y_va, (p_va >= 0.5).astype(int), average='macro', zero_division=0)
        historico.append(dict(epoca=ep, perda_treino=soma/max(1,len(ld_tr)),
                              macro_f1_val=float(f1_va)))
        if verboso:
            print(f'    ep{ep}  loss={soma/max(1,len(ld_tr)):.4f}  macroF1_val={f1_va:.4f}')

        if f1_va > melhor_f1 + 1e-4:
            melhor_f1, melhor_ep, paciencia = f1_va, ep, 0
            melhor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
        else:
            paciencia += 1
            if paciencia >= CFG['paciencia']:
                if verboso: print(f'    early stopping at epoch {ep} (best: {melhor_ep})')
                break

    if melhor_estado is not None:
        modelo.load_state_dict(melhor_estado)
    p_va, y_va = inferir_probs(modelo, ld_va)
    return modelo, dict(melhor_epoca=melhor_ep, macro_f1_val=float(melhor_f1),
                        historico=historico, probs_val=p_va, y_val=y_va,
                        lr=lr, warmup_ratio=warmup_ratio, epocas_max=epocas_max)


def colapsou(sigla_fonte, info):
    # Degenerate collapse: validation macro-F1 does not exceed the source trivial
    # floor by a minimum margin, i.e. the model converges to a single class.
    piso = CORPORA[sigla_fonte]['baseline_majoritaria']
    return info['macro_f1_val'] < piso + CFG['margem_colapso']


def ajustar_fino_robusto(sigla_fonte, semente, desacentuar=False, verboso=True):
    # XLM-R collapses more often than mBERT on small corpora. When this happens the
    # run is REPEATED with a lower learning rate, longer warm-up and more epochs
    # (Mosbach et al., 2021). The rule was not triggered in the reported run.
    modelo, info = ajustar_fino(sigla_fonte, semente, desacentuar, verboso)
    info['colapso_detectado'] = False
    info['recuperado'] = False

    if colapsou(sigla_fonte, info):
        piso = CORPORA[sigla_fonte]['baseline_majoritaria']
        print(f'    ⚠ COLLAPSE: macroF1_val={info["macro_f1_val"]:.4f} '
              f'vs trivial floor {piso:.4f}. Rerunning with '
              f"lr={CFG['lr_recuperacao']}, warm-up={CFG['warmup_recuperacao']}.")
        del modelo; limpar_gpu()
        modelo, info2 = ajustar_fino(
            sigla_fonte, semente, desacentuar, verboso,
            lr=CFG['lr_recuperacao'], warmup_ratio=CFG['warmup_recuperacao'],
            epocas_max=CFG['epocas_recuperacao'])
        info2['colapso_detectado'] = True
        info2['recuperado'] = not colapsou(sigla_fonte, info2)
        info2['macro_f1_val_antes'] = info['macro_f1_val']
        if not info2['recuperado']:
            print('    ⚠⚠ COLLAPSE PERSISTS after recovery. This seed must NOT')
            print('        be reported as model performance; it is recorded')
            print('        in colapso_detectado.')
        else:
            print(f'    ✓ Recovered: macroF1_val={info2["macro_f1_val"]:.4f}')
        info = info2

    return modelo, info

print('Fine-tuning routine defined (with collapse recovery).')

## 10. Threshold policy and metrics

Each cell stores macro-F1 under the training-time policy (threshold calibrated on source validation
only for in-corpus cells with at least 100 validation examples, 0.5 elsewhere), macro-F1 at a fixed
0.5, and ROC-AUC. The prevalence-matched and oracle regimes are computed in Section 17.

In [ ]:
def grade_candidata(probs, n=512):
    # Threshold candidates taken from the QUANTILES of the probabilities themselves.
    #
    # A fixed linear grid (np.arange(0.02, 0.99, 0.01)) assumes useful thresholds are
    # spread evenly over [0,1]. Fine-tuned transformers are overconfident: the
    # probability mass sits near 0 and 1, and the optimal threshold can be 0.9987,
    # which a linear grid never visits. The oracle threshold (an upper bound by
    # definition) could then fall BELOW the prevalence-matched threshold, which is
    # impossible. A quantile grid is dense where the scores are.
    q = np.quantile(probs, np.linspace(0.0005, 0.9995, n))
    g = np.unique(np.concatenate([q, np.array([0.5]), np.linspace(0.01, 0.99, 99)]))
    # Infinitesimal offset: the cut falls BETWEEN two adjacent scores,
    # avoiding ties when the distribution has point masses.
    return np.clip(g - 1e-9, 0.0, 1.0)

def escolher_limiar(p_val, y_val, grade=None):
    # Calibration on SOURCE validation. Never on test.
    grade = grade_candidata(p_val) if grade is None else grade
    melhor, melhor_f1 = 0.5, -1.0
    for t in grade:
        f = f1_score(y_val, (p_val >= t).astype(int), average='macro', zero_division=0)
        if f > melhor_f1: melhor, melhor_f1 = float(t), f
    return melhor

def limiar_da_politica(src, tgt, limiar_calibrado, n_val):
    if CFG['politica_limiar'] == '05':
        return 0.5, 'fixo_0.5'
    mesma_lingua = CORPORA[src]['lang'] == CORPORA[tgt]['lang']
    intra_dominio = (src == tgt)
    if intra_dominio and n_val >= CFG['min_val_calib']:
        return limiar_calibrado, 'calibrado_val_fonte'
    if not mesma_lingua:
        return 0.5, 'fixo_0.5_cross_lingual'
    return 0.5, 'fixo_0.5_cross_dominio'

def metricas(y, p, limiar):
    yhat = (p >= limiar).astype(int)
    pr, rc, f1, sup = precision_recall_fscore_support(
        y, yhat, labels=[0,1], zero_division=0)
    saida = dict(
        macro_f1 = float(f1_score(y, yhat, average='macro', zero_division=0)),
        f1_neg = float(f1[0]), f1_pos = float(f1[1]),
        prec_neg = float(pr[0]), prec_pos = float(pr[1]),
        rec_neg = float(rc[0]),  rec_pos = float(rc[1]),
        suporte_neg = int(sup[0]), suporte_pos = int(sup[1]),
        acuracia = float(accuracy_score(y, yhat)),
        taxa_predicao_positiva = float(yhat.mean()),
        limiar = float(limiar),
    )
    if len(np.unique(y)) > 1:
        saida['auc_roc'] = float(roc_auc_score(y, p))
        saida['auc_pr']  = float(average_precision_score(y, p))
    else:
        saida['auc_roc'] = saida['auc_pr'] = float('nan')
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
    saida['matriz_confusao'] = dict(vn=int(tn), fp=int(fp), fn=int(fn), vp=int(tp))
    return saida

print('Metrics and threshold policy defined.')

## 11. Running the 6×6 matrix

For each source corpus and seed: one fine-tuning run, then evaluation on the six test partitions.
Probabilities are saved per cell and seed.

In [ ]:
SIGLAS = ['PT1','PT2','PT3','EN','ID','KZ']
RESULTADOS = []
PRED_DIR = os.path.join(OUT_DIR, f'predicoes_M4_{TAG}')
os.makedirs(PRED_DIR, exist_ok=True)

# Test loaders built once and reused by every model.
LOADERS_TESTE = {k: montar_loader(montar_dataset(k, 'teste')) for k in SIGLAS}
# In the orthographic ablation the test set is also de-accented: training on text
# without accents and evaluating on accented text would add a train/test mismatch
# that does not exist in the original setting.
LOADERS_TESTE_NOACC = None

def executar_fonte(sigla_fonte, semente, sufixo=''):
    print(f"\n{'='*70}")
    print(f'  SOURCE {sigla_fonte}{sufixo}  ·  seed {semente}')
    print(f"{'='*70}")

    with cronometro(f'treino_{sigla_fonte}{sufixo}_s{semente}'):
        modelo, info = ajustar_fino_robusto(sigla_fonte, semente,
                                            desacentuar=bool(sufixo))

    limiar_cal = escolher_limiar(info['probs_val'], info['y_val'])
    n_val = len(info['y_val'])
    print(f"  best epoch={info['melhor_epoca']}  macroF1_val={info['macro_f1_val']:.4f}  "
          f'calibrated threshold={limiar_cal:.2f}')

    for tgt in SIGLAS:
        loaders = LOADERS_TESTE if not sufixo else LOADERS_TESTE_NOACC
        with cronometro(f'avaliacao_{sigla_fonte}{sufixo}->{tgt}_s{semente}', verboso=False):
            p, y = inferir_probs(modelo, loaders[tgt])

        lim, origem_lim = limiar_da_politica(sigla_fonte, tgt, limiar_cal, n_val)
        m_pol = metricas(y, p, lim)
        m_05  = metricas(y, p, 0.5)

        reg = dict(
            modelo='M4_XLMR', checkpoint=CFG['checkpoint'],
            variante='base' if not sufixo else 'sem_acento',
            treino=sigla_fonte, teste=tgt, semente=semente,
            grupo=grupo_cenario(sigla_fonte, tgt),
            cenario='intra' if sigla_fonte == tgt else 'transferencia',
            mesma_lingua=(CORPORA[sigla_fonte]['lang'] == CORPORA[tgt]['lang']),
            n_teste=int(len(y)),
            baseline_majoritaria=CORPORA[tgt]['baseline_majoritaria'],
            macro_f1=m_pol['macro_f1'], limiar=m_pol['limiar'], origem_limiar=origem_lim,
            macro_f1_05=m_05['macro_f1'],
            auc_roc=m_05['auc_roc'], auc_pr=m_05['auc_pr'],
            melhor_epoca=info['melhor_epoca'], macro_f1_val=info['macro_f1_val'],
            colapso_detectado=info['colapso_detectado'],
            recuperado=info['recuperado'],
            lr_usada=info['lr'], warmup_usado=info['warmup_ratio'],
        )
        for k, v in m_05.items():
            if k not in ('macro_f1','limiar'):
                reg[f'{k}_05'] = v
        reg['ganho_sobre_piso_05'] = reg['macro_f1_05'] - reg['baseline_majoritaria']
        RESULTADOS.append(reg)

        np.savez_compressed(
            os.path.join(PRED_DIR, f'{sigla_fonte}{sufixo}_to_{tgt}_s{semente}.npz'),
            probs=p, y=y)

        marca = '✓' if reg['ganho_sobre_piso_05'] > 0 else '✗ BELOW FLOOR'
        print(f"    → {tgt:<4} macroF1(0.5)={reg['macro_f1_05']:.3f}  "
              f"AUC={reg['auc_roc']:.3f}  floor={reg['baseline_majoritaria']:.3f}  {marca}")

    del modelo; limpar_gpu()

with cronometro('MATRIZ_COMPLETA'):
    for semente in CFG['sementes']:
        for fonte in SIGLAS:
            executar_fonte(fonte, semente)

print(f'\nDone: {len(RESULTADOS)} records '
      f"({len(SIGLAS)}×{len(SIGLAS)} cells × {len(CFG['sementes'])} seeds).")

_dfc = pd.DataFrame(RESULTADOS)
_col = _dfc[_dfc.colapso_detectado][['treino','semente','recuperado']].drop_duplicates()
if len(_col):
    print(f'\n⚠ Collapse detected in {len(_col)} run(s):')
    print(_col.to_string(index=False))
    print('  Reruns with a lower learning rate (recorded in colapso_detectado).')
    if (~_col.recuperado).any():
        print('  ⚠⚠ Some run did NOT recover.')
else:
    print('\nNo fine-tuning collapse detected.')

## 12. Aggregation across seeds

Each cell is reported as the mean and standard deviation over the three seeds.

In [ ]:
DF = pd.DataFrame(RESULTADOS)
DF_BASE = DF[DF['variante'] == 'base']

AGG = (DF_BASE
       .groupby(['treino','teste','grupo','cenario','mesma_lingua',
                 'baseline_majoritaria','n_teste'], as_index=False)
       .agg(macro_f1_05_media=('macro_f1_05','mean'),
            macro_f1_05_dp   =('macro_f1_05','std'),
            macro_f1_pol_media=('macro_f1','mean'),
            auc_media=('auc_roc','mean'), auc_dp=('auc_roc','std'),
            f1_pos_media=('f1_pos_05','mean'), f1_neg_media=('f1_neg_05','mean'),
            rec_pos_media=('rec_pos_05','mean'),
            taxa_pred_pos_media=('taxa_predicao_positiva_05','mean'),
            n_sementes=('semente','nunique')))
AGG['ganho_sobre_piso'] = AGG['macro_f1_05_media'] - AGG['baseline_majoritaria']

pd.set_option('display.width', 200, 'display.max_columns', 50)
print('── Fine-tuning noise (standard deviation across seeds) ──')
diag  = AGG[AGG.treino == AGG.teste][['treino','macro_f1_05_media','macro_f1_05_dp']]
print(diag.to_string(index=False))

dp_diag   = float(diag['macro_f1_05_dp'].median())
transf    = AGG[AGG.treino != AGG.teste]
dp_transf = float(transf['macro_f1_05_dp'].median())
dp_max    = float(transf['macro_f1_05_dp'].max())
pior      = transf.loc[transf['macro_f1_05_dp'].idxmax()]

# dp_tip is the noise reference used in the next sections. It MUST come from the
# transfer cells: estimating noise from the diagonal underestimates the
# uncertainty of exactly the cells the paper interprets.
dp_tip = dp_transf

print(f'\n  median sd on the diagonal ........ {dp_diag:.4f}')
print(f'  median sd on transfer cells ...... {dp_transf:.4f}  '
      f'({dp_transf/max(dp_diag,1e-9):.1f}x)')
print(f"  worst cell ...................... {dp_max:.4f}  "
      f"({pior['treino']}→{pior['teste']})")
print(f'\n→ Noise reference for comparisons between cells: ~{2*dp_transf:.3f}')
print('  Initialisation variance in zero-shot transfer is much larger than')
print('  in in-corpus evaluation. The sd is reported PER CELL rather than as a single')
print('  value: sources with few training examples have an sd of a different order')
print('  and do not support fine comparisons.')

instaveis = transf[transf['macro_f1_05_dp'] > 0.05]
if len(instaveis):
    print('\n  ⚠ Cells with sd > 0.05 — not used for comparative claims:')
    for _, r in instaveis.iterrows():
        print(f"      {r['treino']}→{r['teste']}  dp={r['macro_f1_05_dp']:.3f}")

print('\n── Full matrix ──')
cols = ['treino','teste','grupo','macro_f1_05_media','macro_f1_05_dp','auc_media',
        'baseline_majoritaria','ganho_sobre_piso','rec_pos_media','taxa_pred_pos_media']
print(AGG.sort_values(['treino','teste'])[cols].to_string(index=False,
      float_format=lambda v: f'{v:6.3f}'))

# ── Surface shortcuts x diagonal separability ───────────────────────────────
# How much of the separability reached on the diagonal was already available
# without reading the content? A high share means the diagonal of that corpus
# measures provenance rather than the task.
SOND['auc_diagonal'] = [float(AGG[(AGG.treino==k)&(AGG.teste==k)]['auc_media'].iloc[0])
                        for k in SOND['corpus']]
SOND['fracao_explicada'] = ((SOND['auc_superficial'] - 0.5) /
                            (SOND['auc_diagonal'] - 0.5)).clip(lower=0)

print('\n── Surface shortcut x diagonal ──')
print(SOND[['corpus','auc_superficial','auc_diagonal','fracao_explicada',
            'rho_comprimento']].to_string(index=False,
            float_format=lambda v: f'{v:7.3f}'))
suspeitos = SOND[SOND.fracao_explicada > 0.25]
if len(suspeitos):
    print('\n  ⚠ Diagonal with a surface component above 25%:')
    for _, r in suspeitos.iterrows():
        print(f"      {r['corpus']}: {r['fracao_explicada']*100:.0f}% of the separability")
    print('    The diagonal of these corpora is not a clean reference for ΔF1;')
    print('    retention (Section 17) normalises by the floor of the target')
    print('    and is used as the main measure.')

# ── Check of the inversions predicted by the sign of rho ────────────────────
if DIAGNOSTICOS.get('pares_rho_oposto'):
    print('\n── Predicted x observed inversion ──')
    for p in DIAGNOSTICOS['pares_rho_oposto']:
        a, b = p['a'], p['b']
        for x, y_ in ((a, b), (b, a)):
            auc = float(AGG[(AGG.treino==x)&(AGG.teste==y_)]['auc_media'].iloc[0])
            veredito = 'INVERTED ✓ as predicted' if auc < 0.5 else 'not inverted'
            print(f'  {x}→{y_}  AUC={auc:.3f}  {veredito}')
    print('\n  Confirmed inversion = the model learned the length shortcut and')
    print('  applied it to a target where the rule is reversed: a corpus construction')
    print('  artefact, not a distance between languages.')

## 13. Exploratory ΔF1 decomposition (fixed threshold)

Exploratory decomposition of the drop in macro-F1 into language and domain contrasts at a fixed 0.5
threshold. The paper does not use this table: it reports retention-based costs computed by the
scripts in `analysis/` (see the repository README).

In [ ]:
def f1_celula(src, tgt, coluna='macro_f1_05_media'):
    linha = AGG[(AGG.treino == src) & (AGG.teste == tgt)]
    return float(linha[coluna].iloc[0]) if len(linha) else float('nan')

# Domain-similarity annotation (declared by the authors, reported in the paper)
DOMINIO_SIMILAR = {
    ('PT1','ID') : 'político, rede social',
    ('ID','PT1') : 'político, rede social',
    ('PT2','EN') : 'comentário/microblog geral',
    ('EN','PT2') : 'comentário/microblog geral',
    # PT3 (ToLD-Br) and EN (Davidson) are both general-topic Twitter: the pair that
    # keeps the domain closest across different languages in the whole matrix.
    ('PT3','EN') : 'Twitter, geral',
    ('EN','PT3') : 'Twitter, geral',
}

def decomposicao(src):
    intra = f1_celula(src, src)
    linhas = []
    for tgt in SIGLAS:
        if tgt == src: continue
        delta = intra - f1_celula(src, tgt)
        mesma = CORPORA[src]['lang'] == CORPORA[tgt]['lang']
        if mesma:
            tipo = 'domínio apenas'
        elif (src, tgt) in DOMINIO_SIMILAR:
            tipo = 'língua (domínio ~ constante)'
        else:
            tipo = 'língua + domínio'
        linhas.append(dict(fonte=src, alvo=tgt, tipo=tipo,
                           grupo=grupo_cenario(src, tgt),
                           f1_intra=intra, f1_transf=f1_celula(src, tgt),
                           delta_f1=delta,
                           ganho_sobre_piso=f1_celula(src, tgt, 'ganho_sobre_piso'),
                           auc=f1_celula(src, tgt, 'auc_media')))
    return pd.DataFrame(linhas)

DEC = pd.concat([decomposicao(s) for s in SIGLAS], ignore_index=True)
print(DEC.to_string(index=False, float_format=lambda v: f'{v:6.3f}'))

# ── Additive decomposition where the design allows it ───────────────────────
print('\n' + '='*72)
print('  ADDITIVE DECOMPOSITION — source PT1 (only source with separable domain and language)')
print('='*72)
intra   = f1_celula('PT1','PT1')
# The domain cost is the MEAN over the Portuguese targets, not a single cell.
alvos_dom = [t for t in SIGLAS
             if t != 'PT1' and LINGUA_SIGLA[t] == LINGUA_SIGLA['PT1']]
c_dom   = intra - np.mean([f1_celula('PT1', t) for t in alvos_dom])
c_ling  = intra - f1_celula('PT1','ID')      # domain ~ fixed, language changes
print(f'  domain cost computed over: ' +
      ', '.join(f'PT1->{t}' for t in alvos_dom))
for alvo in ['EN','KZ']:
    c_tot = intra - f1_celula('PT1', alvo)
    inter = c_tot - (c_dom + c_ling)
    print(f'\n  PT1 → {alvo}')
    print(f'    ΔF1 total .................. {c_tot:+.3f}')
    print(f'    domain cost (PT1→PT2) ..... {c_dom:+.3f}')
    print(f'    language cost  (PT1→ID) .. {c_ling:+.3f}')
    print(f'    interaction (residual) ..... {inter:+.3f}')
    if abs(inter) > 2 * dp_tip:
        print('    → interaction above noise.')
        print('      NOTE: strong sub-additivity (very negative interaction) is usually a')
        print('      FLOOR EFFECT — transfer has already hit the trivial floor and has')
        print('      nowhere further to fall. Check that the cells involved are above the')
        print('      floor (column ganho_sobre_piso) before reading it as a finding.')
    else:
        print('    → interaction within noise: additivity is not rejected.')

print('\n── Means by design group ──')
print(DEC.groupby('grupo')[['delta_f1','ganho_sobre_piso','auc']]
        .agg(['mean','count']).round(3).to_string())

## 14. Visualisation: macro-F1 and AUC heat maps

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def matriz(coluna):
    M = pd.DataFrame(index=SIGLAS, columns=SIGLAS, dtype=float)
    for _, r in AGG.iterrows():
        M.loc[r['treino'], r['teste']] = r[coluna]
    return M.astype(float)

fig, axes = plt.subplots(1, 3, figsize=(19, 5.2))
paineis = [
    ('macro_f1_05_media', 'Macro-F1 (threshold 0.5)',        'RdYlGn', 0.3, 1.0),
    ('ganho_sobre_piso',  'Gain over the target trivial floor','RdYlGn', -0.3, 0.4),
    ('auc_media',         'AUC-ROC',                           'RdYlGn', 0.4, 1.0),
]
for ax, (col, titulo, cmap, vmin, vmax) in zip(axes, paineis):
    sns.heatmap(matriz(col), annot=True, fmt='.3f', cmap=cmap, vmin=vmin, vmax=vmax,
                center=0 if col == 'ganho_sobre_piso' else None,
                linewidths=0.5, linecolor='white', ax=ax, cbar_kws={'shrink':0.8})
    ax.set_title(titulo, fontsize=11, pad=10)
    ax.set_xlabel('Evaluation corpus'); ax.set_ylabel('Training corpus')
fig.suptitle('M4 — XLM-RoBERTa · mean of '
             f"{len(CFG['sementes'])} seeds", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'M4_matrizes_{TAG}.png'), dpi=200, bbox_inches='tight')
plt.show()

# Macro-F1 x AUC scatter: separates calibration loss from representation loss
fig, ax = plt.subplots(figsize=(6.5, 5.5))
cores = {'diagonal':'#1f3a68','A':'#2e8b57','B':'#d99000','C':'#b22222'}
for g, sub in AGG.groupby('grupo'):
    ax.scatter(sub['auc_media'], sub['macro_f1_05_media'], s=70,
               label=g, color=cores.get(g,'gray'), edgecolor='white', zorder=3)
ax.axhline(0.5, ls='--', lw=1, color='gray')
ax.axvline(0.5, ls='--', lw=1, color='gray')
ax.set_xlabel('ROC-AUC (threshold-independent)')
ax.set_ylabel('Macro-F1 (threshold 0.5)')
ax.set_title('Preserved ranking × calibrated decision', fontsize=11)
ax.legend(title='Group', frameon=False)
plt.tight_layout(); plt.show()

print('Lower-right quadrant (high AUC, low macro-F1) = the model RANKS the target well')
print('but the 0.5 threshold is misplaced: a calibration failure, not a representation failure.')
print('Lower-left quadrant = genuine representation failure.')

## 15. Orthographic ablation: PT1 without accents

OffComBR-3 (PT2) has no accents, while HateBR (PT1) keeps standard spelling. The model is retrained on
de-accented PT1 (source `PT1_noacc`) and evaluated on de-accented targets, to separate the domain
effect from the orthographic difference.

In [ ]:
if CFG['ablacao_desacentuar_pt']:
    LOADERS_TESTE_NOACC = {k: montar_loader(montar_dataset(k, 'teste', desacentuar=True))
                           for k in SIGLAS}
    with cronometro('ABLACAO_ortografica'):
        for semente in CFG['sementes']:
            executar_fonte('PT1', semente, sufixo='_noacc')

    DF = pd.DataFrame(RESULTADOS)
    orig = DF[(DF.variante=='base')       & (DF.treino=='PT1')]
    abl  = DF[(DF.variante=='sem_acento') & (DF.treino=='PT1')]

    def m(d, tgt): return d[d.teste==tgt]['macro_f1_05'].mean()

    d_orig = m(orig,'PT1') - m(orig,'PT2')
    d_abl  = m(abl,'PT1')  - m(abl,'PT2')
    print('\n' + '='*70)
    print('  ORTHOGRAPHIC ABLATION — decomposition of the PT1→PT2 cell')
    print('='*70)
    print(f"  PT1→PT1 original ....... {m(orig,'PT1'):.3f}   de-accented: {m(abl,'PT1'):.3f}")
    print(f"  PT1→PT2 original ....... {m(orig,'PT2'):.3f}   de-accented: {m(abl,'PT2'):.3f}")
    print(f'  ΔF1 original ........... {d_orig:+.3f}')
    print(f'  ΔF1 with harmonised spelling ..... {d_abl:+.3f}')
    print(f'  cost attributable to spelling .... {d_orig - d_abl:+.3f}')
    DIAGNOSTICOS['ablacao_ortografica'] = dict(
        delta_original=float(d_orig), delta_harmonizado=float(d_abl),
        custo_ortografico=float(d_orig - d_abl))
    if abs(d_orig - d_abl) > 2*dp_tip:
        print('\n  → A relevant part of the "domain cost" is orthographic.')
        print('    The harmonised domain ΔF1 is the one to read,')
        print('    with the original as a secondary value.')
    else:
        print('\n  → Orthographic effect within noise: the measured cost is a domain cost.')
else:
    print('Ablation disabled in CFG.')

## 16. Uncertainty: test sampling and initialisation

Two sources reported separately: bootstrap over the test indices (resampled once and shared by the
three seeds) and the standard deviation across seeds. `incerteza_total` combines them in quadrature.

In [ ]:
def carregar_predicoes(src, tgt, variante=''):
    dados = []
    for s in CFG['sementes']:
        arq = os.path.join(PRED_DIR, f'{src}{variante}_to_{tgt}_s{s}.npz')
        if os.path.exists(arq):
            d = np.load(arq)
            dados.append((d['probs'], d['y']))
    if not dados:
        return None
    y0 = dados[0][1]
    for _, y in dados[1:]:
        assert np.array_equal(y, y0), (
            f'{src}->{tgt}: test labels differ across seeds. '
            'Partitions should be fixed — check pibic/splits/.')
    return dados

def ic_bootstrap_multissemente(src, tgt, limiar=0.5, n=2000, semente=7):
    dados = carregar_predicoes(src, tgt)
    if dados is None:
        return None
    y0 = dados[0][1]
    rng = np.random.RandomState(semente)
    N = len(y0); vals = []
    for _ in range(n):
        i = rng.randint(0, N, N)
        if len(np.unique(y0[i])) < 2:
            continue
        vals.append(np.mean([
            f1_score(y[i], (p[i] >= limiar).astype(int), average='macro', zero_division=0)
            for p, y in dados]))
    v = np.array(vals)
    return dict(media=float(v.mean()),
                ic_baixo=float(np.percentile(v, 2.5)),
                ic_alto=float(np.percentile(v, 97.5)))

with cronometro('bootstrap_multissemente'):
    ICS = {}
    for src in SIGLAS:
        for tgt in SIGLAS:
            r = ic_bootstrap_multissemente(src, tgt)
            if r:
                linha = AGG[(AGG.treino == src) & (AGG.teste == tgt)].iloc[0]
                r['dp_semente'] = float(linha['macro_f1_05_dp'])
                dp_amo = (r['ic_alto'] - r['ic_baixo']) / (2 * 1.96)
                r['dp_amostragem'] = float(dp_amo)
                r['incerteza_total'] = float(1.96 * np.sqrt(dp_amo**2 + r['dp_semente']**2))
                r['n_teste'] = int(linha['n_teste'])
                ICS[f'{src}->{tgt}'] = r

DIAGNOSTICOS['ic_bootstrap'] = ICS

print(f"{'Cell':<12} {'mean':>7} {'95% CI (sampling)':>22} {'seed sd':>11} "
      f"{'± total':>9} {'n':>7}")
print('-' * 76)
for src in SIGLAS:
    for tgt in SIGLAS:
        ch = f'{src}->{tgt}'
        if ch not in ICS: continue
        r = ICS[ch]
        print(f"{ch:<12} {r['media']:>7.3f} "
              f"[{r['ic_baixo']:>6.3f}, {r['ic_alto']:>6.3f}] "
              f"{r['dp_semente']:>11.3f} {r['incerteza_total']:>9.3f} {r['n_teste']:>7,}")

largas = {k: v for k, v in ICS.items() if v['incerteza_total'] > 0.08}
print(f'\n{len(largas)} of {len(ICS)} cells with total margin above 0.08:')
print('  ' + ', '.join(sorted(largas)))
print('\nThese cells are reported with an explicit margin and do not support')
print('ranking claims among themselves. The cause is test-set size (PT2, ID)')
print('or fine-tuning instability (sources with few training examples).')

## 17. Three threshold regimes and retention

| Regime | Assumption about the target | Role |
|---|---|---|
| R1 · fixed threshold 0.5 | none | deployment without target information |
| R2 · prevalence-matched threshold | knows only the base rate, never the individual labels | main regime of the paper |
| R3 · oracle threshold | knows the target labels | upper bound, not a result |

R3 − R1 is the calibration cost of a cell. Retention normalises the gain over the trivial floor of
the target by the gain of the source on its own diagonal. Everything runs on the saved `.npz` files.

In [ ]:
def limiar_por_prevalencia(probs, prevalencia_alvo):
    # Cut that makes the predicted positive rate equal the target prevalence.
    # Uses only the base rate, never the individual labels.
    return float(np.quantile(probs, 1.0 - prevalencia_alvo))

def limiar_oraculo(probs, y, grade=None):
    # UPPER BOUND, not a result. Uses the same quantile grid as escolher_limiar:
    # with a linear grid the bound could fall below R2, which is impossible.
    grade = grade_candidata(probs) if grade is None else grade
    melhor, melhor_f1 = 0.5, -1.0
    for t in grade:
        f = f1_score(y, (probs >= t).astype(int), average='macro', zero_division=0)
        if f > melhor_f1:
            melhor, melhor_f1 = float(t), f
    return melhor

with cronometro('regimes_limiar'):
    linhas = []
    for src in SIGLAS:
        for tgt in SIGLAS:
            dados = carregar_predicoes(src, tgt)
            if dados is None: continue
            prev = CORPORA[tgt]['prev_pos']
            r1, r2, r3 = [], [], []
            for p, y in dados:
                r1.append(f1_score(y, (p >= 0.5).astype(int),
                                   average='macro', zero_division=0))
                r2.append(f1_score(y, (p >= limiar_por_prevalencia(p, prev)).astype(int),
                                   average='macro', zero_division=0))
                r3.append(f1_score(y, (p >= limiar_oraculo(p, y)).astype(int),
                                   average='macro', zero_division=0))
            ag = AGG[(AGG.treino == src) & (AGG.teste == tgt)].iloc[0]
            linhas.append(dict(
                treino=src, teste=tgt, grupo=grupo_cenario(src, tgt),
                piso=float(CORPORA[tgt]['baseline_majoritaria']),
                R1_fixo=float(np.mean(r1)),
                R1_dp=float(np.std(r1, ddof=1)) if len(r1) > 1 else 0.0,
                R2_prevalencia=float(np.mean(r2)),
                R2_dp=float(np.std(r2, ddof=1)) if len(r2) > 1 else 0.0,
                R3_oraculo=float(np.mean(r3)),
                auc=float(ag['auc_media'])))

REG = pd.DataFrame(linhas)
REG['custo_calibracao'] = REG['R3_oraculo'] - REG['R1_fixo']
REG['R2_sobre_piso']    = REG['R2_prevalencia'] - REG['piso']

# Sanity check: R3 is an upper bound and MUST dominate R1 and R2 in every cell.
viol = REG[(REG.R3_oraculo < REG.R1_fixo - 1e-6) |
           (REG.R3_oraculo < REG.R2_prevalencia - 1e-6)]
print(f'Violations of "R3 is an upper bound": {len(viol)} of {len(REG)}')
if len(viol):
    print(viol[['treino','teste','R1_fixo','R2_prevalencia','R3_oraculo']]
          .to_string(index=False))
    print('→ VIOLATIONS REMAIN: increase n in grade_candidata(). While they remain,')
    print('  the custo_calibracao column cannot be interpreted.')
else:
    print('→ None. The custo_calibracao column can be interpreted.\n')

print('='*92)
print('  THREE THRESHOLD REGIMES')
print('='*92)
print(REG.sort_values(['treino','teste'])[
    ['treino','teste','grupo','piso','R1_fixo','R2_prevalencia','R3_oraculo',
     'custo_calibracao','R2_sobre_piso','auc']
    ].to_string(index=False, float_format=lambda v: f'{v:6.3f}'))

print('\n── Means by group ──')
print(REG.groupby('grupo')[['R1_fixo','R2_prevalencia','R3_oraculo',
                            'custo_calibracao','auc']].mean().round(3).to_string())

In [ ]:
# ── Calibration x representation ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5.6))
cores = {'diagonal':'#1f3a68','A':'#2e8b57','B':'#d99000','C':'#b22222'}
for g, sub in REG.groupby('grupo'):
    ax.scatter(sub['auc'], sub['custo_calibracao'], s=80, label=g,
               color=cores.get(g,'gray'), edgecolor='white', zorder=3)
for _, r in REG.iterrows():
    if r['grupo'] != 'diagonal':
        ax.annotate(f"{r['treino']}→{r['teste']}", (r['auc'], r['custo_calibracao']),
                    fontsize=7, alpha=0.75, xytext=(4,3), textcoords='offset points')
ax.axvline(0.5, ls='--', lw=1, color='gray')
ax.axhline(0.0, ls='--', lw=1, color='gray')
ax.set_xlabel('ROC-AUC  →  quality of the transferred representation')
ax.set_ylabel('Calibration cost (R3 − R1)')
ax.set_title('Calibration loss × representation loss', fontsize=11)
ax.legend(title='Group', frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'M4_calibracao_vs_representacao_{TAG}.png'),
            dpi=200, bbox_inches='tight')
plt.show()
print('High and to the right = the representation transferred, only the threshold is misplaced.')
print('Low and to the left = representation failure; no threshold recovers the cell.')

# ── ΔF1 decomposition under the three regimes ───────────────────────────────
def f1_reg(src, tgt, col):
    l = REG[(REG.treino == src) & (REG.teste == tgt)]
    return float(l[col].iloc[0]) if len(l) else float('nan')

print('\n' + '='*92)
print('  ΔF1 DECOMPOSITION — source PT1, under the three regimes')
print('='*92)
razoes = {}
for col, rot in [('R1_fixo','R1 · fixed threshold 0.5'),
                 ('R2_prevalencia','R2 · prevalence-matched  ← main'),
                 ('R3_oraculo','R3 · oracle (upper bound)')]:
    intra  = f1_reg('PT1','PT1', col)
    alvos_dom = [t for t in SIGLAS
                 if t != 'PT1' and LINGUA_SIGLA[t] == LINGUA_SIGLA['PT1']]
    c_dom  = intra - float(np.mean([f1_reg('PT1', t, col) for t in alvos_dom]))
    c_ling = intra - f1_reg('PT1','ID',  col)   # domain ~ fixed, language changes
    razao  = c_dom / c_ling if c_ling > 0.01 else float('nan')
    razoes[col] = razao
    print(f'\n  {rot}   (intra = {intra:.3f})')
    print(f'    domain cost (PT1→PT2) ..... {c_dom:+.3f}')
    print(f'    language cost  (PT1→ID) ... {c_ling:+.3f}')
    print(f'    domain/language ratio ..... {razao:.2f}')

print('\n' + '-'*92)
if not np.isnan(razoes['R1_fixo']) and not np.isnan(razoes['R2_prevalencia']):
    queda = razoes['R1_fixo'] - razoes['R2_prevalencia']
    print(f'  Change of the domain/language ratio from R1 to R2: {queda:+.2f}')
    if abs(queda) > 0.3:
        print('  → A substantial part of the "domain cost" measured with a fixed threshold was')
        print('    a base-rate mismatch between HateBR and OffComBR-3 — a property of how')
        print('    the corpora were built, not of the domain. R2 is the main regime; R1 is')
        print('    reported alongside it.')
    else:
        print('  → The ratio is stable across regimes: the domain cost survives the')
        print('    prevalence correction and can be read as a domain effect.')

# ── Retention: comparison across targets with different floors ──────────────
# Raw ΔF1 is not comparable across targets with different prevalence: PT1 has a
# floor of 0.333 and KZ of 0.456, so the same absolute drop means different things.
# Retention normalises by the room available above each floor.
ret = []
for src in SIGLAS:
    base = REG[(REG.treino == src) & (REG.teste == src)]['R2_sobre_piso'].iloc[0]
    for tgt in SIGLAS:
        if tgt == src: continue
        g = REG[(REG.treino == src) & (REG.teste == tgt)]['R2_sobre_piso'].iloc[0]
        ret.append(dict(fonte=src, alvo=tgt, grupo=grupo_cenario(src, tgt),
                        ganho_intra=base, ganho_transf=g,
                        retencao=g/base if base > 0.01 else float('nan')))
RET = pd.DataFrame(ret)
print('\n' + '='*92)
print('  RETENTION OF THE GAIN OVER THE FLOOR (regime R2)')
print('='*92)
print(RET.to_string(index=False, float_format=lambda v: f'{v:7.3f}'))
print('\n── Mean retention by group ──')
print(RET.groupby('grupo')['retencao'].agg(['mean','count']).round(3).to_string())

REG.to_csv(os.path.join(OUT_DIR, f'M4_regimes_limiar_{TAG}.csv'), index=False)
RET.to_csv(os.path.join(OUT_DIR, f'M4_retencao_{TAG}.csv'), index=False)
print(f'\nSaved: M4_regimes_limiar_{TAG}.csv')

## 18. Export for qualitative error analysis

Exports, for selected cells, the most confident errors. The exported files contain corpus texts and are
not included in the repository.

In [ ]:
def exportar_erros(src, tgt, semente=None, k=40):
    semente = semente or CFG['sementes'][0]
    d = np.load(os.path.join(PRED_DIR, f'{src}_to_{tgt}_s{semente}.npz'))
    p, y = d['probs'], d['y']
    idx = CORPORA[tgt]['idx_teste']
    textos = [CORPORA[tgt]['textos'][i] for i in idx]
    yhat = (p >= 0.5).astype(int)
    df = pd.DataFrame(dict(texto=textos, y=y, prob=p, pred=yhat))
    df['erro'] = df.y != df.pred
    df['confianca'] = np.abs(df.prob - 0.5)
    fp = df[(df.erro) & (df.pred==1)].nlargest(k, 'confianca')
    fn = df[(df.erro) & (df.pred==0)].nlargest(k, 'confianca')
    fp['tipo'] = 'falso_positivo'; fn['tipo'] = 'falso_negativo'
    out = pd.concat([fp, fn]); out['fonte'] = src; out['alvo'] = tgt
    return out

CELULAS_ERRO = [('PT1','PT1'), ('PT1','PT2'), ('PT1','PT3'), ('PT2','PT3'),
                ('PT3','PT1'), ('PT1','EN'),  ('EN','PT1'),
                ('PT1','KZ'),  ('KZ','PT1'),  ('PT1','ID')]
with cronometro('exportar_erros'):
    ERROS = pd.concat([exportar_erros(a,b) for a,b in CELULAS_ERRO], ignore_index=True)
arq_erros = os.path.join(OUT_DIR, f'M4_analise_erro_{TAG}.csv')
ERROS.to_csv(arq_erros, index=False)
print(f'{len(ERROS)} examples exported to {arq_erros}')

print('\n── Sample: most confident false positives in PT1→PT2 ──')
amostra = ERROS[(ERROS.fonte=='PT1') & (ERROS.alvo=='PT2') &
                (ERROS.tipo=='falso_positivo')].head(8)
for _, r in amostra.iterrows():
    print(f"  p={r['prob']:.3f} | {str(r['texto'])[:130]}")

## 19. M3 × M4 comparison

Loads the most recent M3 results JSON from `resultados/` and compares the two models cell by cell.
Partitions, seeds and threshold regimes are identical, so the difference is attributable to the model.

In [ ]:
import glob

arquivos_m3 = sorted(glob.glob(os.path.join(OUT_DIR, 'M3_mBERT_resultados_*.json')))
if not arquivos_m3:
    print('No M3 results found in', OUT_DIR)
    print('The comparison needs the M3 JSON. Skip this section or fix the path.')
    CMP = None
else:
    caminho_m3 = arquivos_m3[-1]
    print(f'M3 loaded from: {os.path.basename(caminho_m3)}')
    with open(caminho_m3, encoding='utf-8') as f:
        M3 = json.load(f)

    a3 = pd.DataFrame(M3['agregado'])
    r3 = pd.DataFrame(M3['regimes_limiar'])
    m3 = (a3[['treino','teste','grupo','macro_f1_05_media','macro_f1_05_dp','auc_media']]
          .merge(r3[['treino','teste','R2_prevalencia','R3_oraculo','R2_sobre_piso']],
                 on=['treino','teste'])
          .rename(columns={'macro_f1_05_media':'R1_M3','macro_f1_05_dp':'dp_M3',
                           'auc_media':'auc_M3','R2_prevalencia':'R2_M3',
                           'R3_oraculo':'R3_M3','R2_sobre_piso':'piso_M3'}))

    m4 = (AGG[['treino','teste','grupo','macro_f1_05_media','macro_f1_05_dp','auc_media']]
          .merge(REG[['treino','teste','R2_prevalencia','R2_dp','R3_oraculo','R2_sobre_piso']],
                 on=['treino','teste'])
          .rename(columns={'macro_f1_05_media':'R1_M4','macro_f1_05_dp':'dp_M4',
                           'auc_media':'auc_M4','R2_prevalencia':'R2_M4',
                           'R2_dp':'dpR2_M4','R3_oraculo':'R3_M4',
                           'R2_sobre_piso':'piso_M4'}))

    CMP = m3.merge(m4, on=['treino','teste','grupo'])
    CMP['d_R2']   = CMP['R2_M4'] - CMP['R2_M3']
    CMP['d_auc']  = CMP['auc_M4'] - CMP['auc_M3']
    CMP['d_piso'] = CMP['piso_M4'] - CMP['piso_M3']
    CMP['margem'] = 2 * (CMP['dp_M3'] + CMP['dp_M4'])   # combined noise
    CMP['relevante'] = CMP['d_R2'].abs() > CMP['margem']

    print('\n' + '='*96)
    print('  M3 (mBERT)  ×  M4 (XLM-RoBERTa)   —   regime R2 and AUC')
    print('='*96)
    print(CMP.sort_values(['treino','teste'])[
        ['treino','teste','grupo','R2_M3','R2_M4','d_R2','margem','relevante',
         'auc_M3','auc_M4','d_auc']
        ].to_string(index=False, float_format=lambda v: f'{v:6.3f}'))

    print('\n── Mean gain by group ──')
    print(CMP.groupby('grupo')[['R2_M3','R2_M4','d_R2','auc_M3','auc_M4','d_auc']]
             .mean().round(3).to_string())

    n_rel = int(CMP['relevante'].sum())
    print(f'\nCells with a difference above the combined noise: {n_rel} of {len(CMP)}')

In [ ]:
if CMP is not None:
    print('='*84)
    print('  PREDICTION 1 — the KZ column is where M4 gains most')
    print('='*84)
    por_alvo = (CMP.groupby('teste')[['d_R2','d_auc']].mean()
                  .sort_values('d_R2', ascending=False).round(3))
    print(por_alvo.to_string())
    melhor_alvo = por_alvo.index[0]
    kz_auc = CMP[CMP.teste == 'KZ']['d_auc'].mean()
    print(f'\n  Target with the largest mean gain in R2: {melhor_alvo}')
    print(f'  Mean AUC gain in the KZ column: {kz_auc:+.3f}')
    if melhor_alvo == 'KZ':
        print('  → PREDICTION 1 CONFIRMED: the mBERT bottleneck on Kazakh was pre-training')
        print('    coverage, which the CommonCrawl data of XLM-R relieves.')
        print('')
    else:
        print('  → PREDICTION 1 NOT CONFIRMED: the Kazakh bottleneck is not pre-training')
        print('    volume. Alternative candidates: the corpus construction artefact')
        print('    (Section 5.3) and typological distance.')
        print('')

    print('\n' + '='*84)
    print('  PREDICTION 2 — among the diagonals, PT1 gains most (emoji)')
    print('='*84)
    diag = CMP[CMP.treino == CMP.teste].copy()
    unk_m3 = {k: v['taxa_unk'] for k, v in M3['diagnosticos']['tokenizacao'].items()}
    unk_m4 = {k: v['taxa_unk'] for k, v in DIAGNOSTICOS['tokenizacao_bruto'].items()}
    diag['unk_M3'] = diag['treino'].map(unk_m3)
    diag['unk_M4'] = diag['treino'].map(unk_m4)
    print(diag[['treino','R2_M3','R2_M4','d_R2','margem','unk_M3','unk_M4']]
          .sort_values('d_R2', ascending=False)
          .to_string(index=False, float_format=lambda v: f'{v:7.4f}'))
    melhor_diag = diag.sort_values('d_R2', ascending=False)['treino'].iloc[0]
    print(f'\n  Diagonal with the largest gain: {melhor_diag}')
    if melhor_diag == 'PT1':
        print('  → PREDICTION 2 CONFIRMED: the drop in the unknown-token rate on PT1')
        print('    goes with the gain: vocabulary coverage (emoji) was a real bottleneck.')
    else:
        print('  → PREDICTION 2 NOT CONFIRMED: the 1.41% unknown tokens in HateBR')
        print('    were not a performance bottleneck. The coverage limitation exists and')
        print('    is measurable, but it does not explain the difference between')
        print('    the models.')

    print('\n' + '='*84)
    print('  ΔF1 DECOMPOSITION — M3 × M4, regime R2, source PT1')
    print('='*84)
    def _dec(df, s='R2_M3'):
        i = float(df[(df.treino=='PT1')&(df.teste=='PT1')][s].iloc[0])
        d = i - float(df[(df.treino=='PT1')&(df.teste=='PT2')][s].iloc[0])
        l = i - float(df[(df.treino=='PT1')&(df.teste=='ID')][s].iloc[0])
        return i, d, l, (d/l if l > 0.01 else float('nan'))
    for rot, col in [('M3 (mBERT)     ','R2_M3'), ('M4 (XLM-RoBERTa)','R2_M4')]:
        i, d, l, r = _dec(CMP, col)
        print(f'  {rot}  intra={i:.3f}  domain={d:+.3f}  language={l:+.3f}  ratio={r:.2f}')
    print('\n  If the ratio is stable between the two models, the decomposition is a')
    print('  property of the CORPORA rather than of the model.')
    print('')

    CMP.to_csv(os.path.join(OUT_DIR, f'M3_vs_M4_comparacao_{TAG}.csv'), index=False)
    print(f'\nSaved: M3_vs_M4_comparacao_{TAG}.csv')

In [ ]:
if CMP is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))

    M = pd.DataFrame(index=SIGLAS, columns=SIGLAS, dtype=float)
    for _, r in CMP.iterrows():
        M.loc[r['treino'], r['teste']] = r['d_R2']
    sns.heatmap(M.astype(float), annot=True, fmt='+.3f', cmap='RdYlGn', center=0,
                vmin=-0.15, vmax=0.15, linewidths=0.5, linecolor='white',
                ax=axes[0], cbar_kws={'shrink':0.8})
    axes[0].set_title('Macro-F1 gain, regime R2  (M4 − M3)', fontsize=11)

    A = pd.DataFrame(index=SIGLAS, columns=SIGLAS, dtype=float)
    for _, r in CMP.iterrows():
        A.loc[r['treino'], r['teste']] = r['d_auc']
    sns.heatmap(A.astype(float), annot=True, fmt='+.3f', cmap='RdYlGn', center=0,
                vmin=-0.15, vmax=0.15, linewidths=0.5, linecolor='white',
                ax=axes[1], cbar_kws={'shrink':0.8})
    axes[1].set_title('ROC-AUC gain  (M4 − M3)', fontsize=11)

    for ax in axes:
        ax.set_xlabel('Evaluation corpus'); ax.set_ylabel('Training corpus')
    fig.suptitle('XLM-RoBERTa minus mBERT · same partitions, same seeds',
                 fontsize=12.5, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f'M3_vs_M4_{TAG}.png'), dpi=200, bbox_inches='tight')
    plt.show()
    print('Green = XLM-R improves. Red = it gets worse, which also happens:')
    print('the larger model does not dominate the smaller one in every cell.')

## 20. Construct-validity ablation: strict English label

Under the broad harmonisation the English positive class is `hate` or `offensive_language` (83.2%
positives); only 6.2% of the test partition is hate speech in the strict sense.

**Part A:** the probabilities already saved are re-scored against the strict label (`class == 0`), with
no retraining. **Part B:** M3/M4 are retrained with `EN_ESTRITO` as source (same partitions as `EN`,
offensive examples kept as negatives). Both parts use AUC, which does not depend on prevalence or
threshold.

In [ ]:
# ── Part A · re-analysis under the strict label, no retraining ──────────────
# No GPU needed. Reuses the probabilities saved in PRED_DIR and only swaps the
# label vector. The test set is the same, so the comparison is paired.

import os
def caminho_dado(filename):
    return os.path.join(DATA_DIR, filename)

_df_en = pd.read_csv(caminho_dado('DataIngles.csv'))
_idx_te_en = CORPORA['EN']['idx_teste']
Y_EN_AMPLO   = (_df_en['class'] != 2).astype(int).values[_idx_te_en]
Y_EN_ESTRITO = (_df_en['class'] == 0).astype(int).values[_idx_te_en]

print(f"EN test: n={len(_idx_te_en)}")
print(f"  positives, broad harmonisation : {Y_EN_AMPLO.sum():5d} ({Y_EN_AMPLO.mean():.3f})")
print(f"  positives, strict harmonisation: {Y_EN_ESTRITO.sum():5d} ({Y_EN_ESTRITO.mean():.3f})")
assert ((Y_EN_ESTRITO == 1) <= (Y_EN_AMPLO == 1)).all(), \
    'strict label is not a subset of the broad label'


def _auc(y, p):
    """AUC via Mann-Whitney, with tie correction."""
    y = np.asarray(y); p = np.asarray(p, dtype=float)
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return float('nan')
    ordem = np.argsort(p, kind='mergesort'); ps = p[ordem]
    postos = np.empty(len(p)); i = 0
    while i < len(ps):
        j = i
        while j + 1 < len(ps) and ps[j + 1] == ps[i]:
            j += 1
        postos[ordem[i:j + 1]] = 0.5 * (i + j) + 1
        i = j + 1
    return (postos[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def ablacao_en_alvo(n_boot=1000, semente_boot=7):
    """Compare AUC under the two label definitions, cell by cell."""
    rng = np.random.RandomState(semente_boot)
    linhas = []
    for src in SIGLAS:
        arqs = [os.path.join(PRED_DIR, f'{src}_to_EN_s{s}.npz') for s in CFG['sementes']]
        arqs = [a for a in arqs if os.path.exists(a)]
        if not arqs:
            continue
        probs = []
        for a in arqs:
            z = np.load(a)
            if not np.array_equal(z['y'].astype(int), Y_EN_AMPLO):
                print(f'  {src}: labels in the .npz do not match the corpus — skipping')
                probs = []
                break
            probs.append(z['probs'].astype(float))
        if not probs:
            continue

        aa = float(np.mean([_auc(Y_EN_AMPLO, p) for p in probs]))
        ae = float(np.mean([_auc(Y_EN_ESTRITO, p) for p in probs]))

        n = len(Y_EN_AMPLO); difs = []
        for _ in range(n_boot):
            i = rng.randint(0, n, n)
            ya, ye = Y_EN_AMPLO[i], Y_EN_ESTRITO[i]
            if len(np.unique(ya)) < 2 or len(np.unique(ye)) < 2:
                continue
            difs.append(np.mean([_auc(ye, p[i]) for p in probs])
                        - np.mean([_auc(ya, p[i]) for p in probs]))
        difs = np.array(difs)
        linhas.append(dict(
            modelo=CFG['modelo_id'], fonte=src, cenario='diagonal' if src == 'EN' else 'transferência',
            auc_ampla=aa, auc_estrita=ae, delta=ae - aa,
            ic_baixo=float(np.percentile(difs, 2.5)) if len(difs) else np.nan,
            ic_alto=float(np.percentile(difs, 97.5)) if len(difs) else np.nan))

    df = pd.DataFrame(linhas)
    print(f"\n{'source':7}{'scenario':16}{'AUC broad':>11}{'AUC strict':>13}{'delta':>9}{'95% CI':>22}")
    print('-' * 78)
    for _, r in df.iterrows():
        ic = f"[{r['ic_baixo']:+.3f},{r['ic_alto']:+.3f}]" if pd.notna(r['ic_baixo']) else ''
        print(f"{r['fonte']:7}{r['cenario']:16}{r['auc_ampla']:>11.3f}"
              f"{r['auc_estrita']:>13.3f}{r['delta']:>+9.3f}{ic:>22}")

    diag = df[df.fonte == 'EN']
    if len(diag):
        d = diag.iloc[0]
        print(f"\n  DIAGONAL: AUC {d['auc_ampla']:.3f} under the broad label, "
              f"{d['auc_estrita']:.3f} under the strict one.")
        if d['auc_estrita'] < 0.60:
            print('  The model trained on the broad label does NOT rank strict hate above')
            print('  chance. In-corpus performance on English is about offensiveness and')
            print('  carries no information about hate speech in the strict sense.')
            print('  This is a construct-validity issue for the English target.')
    tr = df[df.fonte != 'EN']
    if len(tr):
        print(f"\n  Transfer: mean delta {tr['delta'].mean():+.3f} "
              f"(min {tr['delta'].min():+.3f}, max {tr['delta'].max():+.3f})")

    df.to_csv(os.path.join(OUT_DIR, f"ablacao_en_alvo_{CFG['modelo_id']}.csv"), index=False)
    return df


ABL_ALVO = ablacao_en_alvo()

### Part B · strict English as source corpus

`EN_ESTRITO` is registered as a seventh corpus with the same train, validation and test indices as
`EN`; only the label vector changes.

In [ ]:
# ── Part B · EN_ESTRITO as source ───────────────────────────────────────────
# GPU needed. Registers a seventh corpus with the SAME partitions as EN and only
# the label vector changed.

_rot_estrito = (_df_en['class'] == 0).astype(int).tolist()
_piso = lambda y: (lambda q: q / (1 + q))(max(np.mean(y), 1 - np.mean(y)))

CORPORA['EN_ESTRITO'] = dict(CORPORA['EN'])          # shallow copy: same texts and indices
CORPORA['EN_ESTRITO']['rotulos'] = _rot_estrito
CORPORA['EN_ESTRITO']['prev_pos'] = float(np.mean(_rot_estrito))
CORPORA['EN_ESTRITO']['baseline_majoritaria'] = float(_piso(_rot_estrito))
CORPORA['EN_ESTRITO']['nome'] = 'Davidson (ódio estrito)'
LINGUA_SIGLA['EN_ESTRITO'] = 'en'

LOADERS_TESTE['EN_ESTRITO'] = montar_loader(montar_dataset('EN_ESTRITO', 'teste'))

print(f"EN_ESTRITO: prevalence {CORPORA['EN_ESTRITO']['prev_pos']:.3f}, "
      f"floor {CORPORA['EN_ESTRITO']['baseline_majoritaria']:.3f}")
print(f"  partitions identical to EN: "
      f"{CORPORA['EN_ESTRITO']['idx_teste'] is CORPORA['EN']['idx_teste']}")

RESULTADOS_ABL = []
ALVOS_ABL = ['PT1', 'PT2', 'PT3', 'ID', 'KZ', 'EN_ESTRITO']


def executar_fonte_estrita(semente):
    with cronometro(f'treino_EN_ESTRITO_s{semente}'):
        modelo, info = ajustar_fino('EN_ESTRITO', semente)
    limiar_cal = escolher_limiar(info['probs_val'], info['y_val'])
    n_val = len(info['y_val'])
    print(f"  best epoch={info['melhor_epoca']}  "
          f"macroF1_val={info['macro_f1_val']:.4f}")

    for tgt in ALVOS_ABL:
        p, y = inferir_probs(modelo, LOADERS_TESTE[tgt])
        lim, origem = limiar_da_politica('EN_ESTRITO', tgt, limiar_cal, n_val)
        m05 = metricas(y, p, 0.5)
        RESULTADOS_ABL.append(dict(
            modelo=CFG['modelo_id'], variante='en_estrito',
            treino='EN_ESTRITO', teste=tgt, semente=semente, n_teste=int(len(y)),
            baseline_majoritaria=CORPORA[tgt]['baseline_majoritaria'],
            macro_f1_05=m05['macro_f1'], auc_roc=m05['auc_roc'],
            macro_f1=metricas(y, p, lim)['macro_f1'], origem_limiar=origem,
            ganho_sobre_piso_05=m05['macro_f1'] - CORPORA[tgt]['baseline_majoritaria'],
            melhor_epoca=info['melhor_epoca']))
        np.savez_compressed(
            os.path.join(PRED_DIR, f'EN_ESTRITO_to_{tgt}_s{semente}.npz'), probs=p, y=y)
        marca = '✓' if RESULTADOS_ABL[-1]['ganho_sobre_piso_05'] > 0 else '✗ BELOW FLOOR'
        print(f"    → {tgt:<11} macroF1(0.5)={m05['macro_f1']:.3f}  "
              f"AUC={m05['auc_roc']:.3f}  {marca}")
    del modelo; limpar_gpu()


with cronometro('ABLACAO_EN_ESTRITO'):
    for semente in CFG['sementes']:
        print(f"\n{'='*70}\n  SOURCE EN_ESTRITO  ·  seed {semente}\n{'='*70}")
        executar_fonte_estrita(semente)

DF_ABL = pd.DataFrame(RESULTADOS_ABL)
DF_ABL.to_csv(os.path.join(OUT_DIR, f"ablacao_en_fonte_{CFG['modelo_id']}.csv"), index=False)
print(f'\n{len(DF_ABL)} ablation records saved.')

### Comparison: English as source under the two label definitions

`EN → target` against `EN_ESTRITO → target` on the five non-English targets; the bootstrap resamples
the same indices for both.

In [ ]:
def comparar_fonte_en(n_boot=1000, semente_boot=7):
    rng = np.random.RandomState(semente_boot)
    linhas = []
    for tgt in ['PT1', 'PT2', 'PT3', 'ID', 'KZ']:
        pares = []
        for s in CFG['sementes']:
            a = os.path.join(PRED_DIR, f'EN_to_{tgt}_s{s}.npz')
            b = os.path.join(PRED_DIR, f'EN_ESTRITO_to_{tgt}_s{s}.npz')
            if os.path.exists(a) and os.path.exists(b):
                za, zb = np.load(a), np.load(b)
                pares.append((za['probs'].astype(float), zb['probs'].astype(float),
                              za['y'].astype(int)))
        if not pares:
            continue
        aa = float(np.mean([_auc(y, pa) for pa, pb, y in pares]))
        ae = float(np.mean([_auc(y, pb) for pa, pb, y in pares]))
        n = len(pares[0][2]); difs = []
        for _ in range(n_boot):
            i = rng.randint(0, n, n)
            if len(np.unique(pares[0][2][i])) < 2:
                continue
            difs.append(np.mean([_auc(y[i], pb[i]) for pa, pb, y in pares])
                        - np.mean([_auc(y[i], pa[i]) for pa, pb, y in pares]))
        difs = np.array(difs)
        linhas.append(dict(modelo=CFG['modelo_id'], alvo=tgt,
                           auc_fonte_ampla=aa, auc_fonte_estrita=ae, delta=ae - aa,
                           ic_baixo=float(np.percentile(difs, 2.5)) if len(difs) else np.nan,
                           ic_alto=float(np.percentile(difs, 97.5)) if len(difs) else np.nan,
                           p=float(2 * min((difs <= 0).mean(), (difs >= 0).mean()))
                           if len(difs) else np.nan))
    if not linhas:
        print('Run Part B first.')
        return None
    df = pd.DataFrame(linhas)
    print(f"{'target':7}{'AUC broad source':>17}{'AUC strict source':>19}{'delta':>9}{'95% CI':>22}{'p':>8}")
    print('-' * 82)
    for _, r in df.iterrows():
        ic = f"[{r['ic_baixo']:+.3f},{r['ic_alto']:+.3f}]" if pd.notna(r['ic_baixo']) else ''
        print(f"{r['alvo']:7}{r['auc_fonte_ampla']:>17.3f}{r['auc_fonte_estrita']:>19.3f}"
              f"{r['delta']:>+9.3f}{ic:>22}{r['p']:>8.3f}")
    print(f"\n  mean delta: {df['delta'].mean():+.3f}")
    print('\n  Reading: a delta close to zero means that the label definition of the SOURCE')
    print('  does not change the quality of the transferred representation — the language')
    print('  cost of the main matrix holds for both definitions. A very negative delta')
    print('  means that training on the strict label, with only 6% positives, gives a')
    print('  worse representation, and the comparison between corpora is limited to the')
    print('  broad construct.')
    df.to_csv(os.path.join(OUT_DIR, f"ablacao_comparacao_fonte_{CFG['modelo_id']}.csv"),
              index=False)
    return df


ABL_FONTE = comparar_fonte_en()

## 21. Results persistence

In [ ]:
TEMPOS_MIN = {k: round(v/60, 3) for k, v in sorted(TEMPOS.items())}

SAIDA = dict(
    modelo='M4_XLMR',
    checkpoint=CFG['checkpoint'],
    gerado_em=datetime.now().isoformat(),
    ambiente=dict(torch=torch.__version__,
                  gpu=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)),
    config=CFG,
    corpora={k: {kk: vv for kk, vv in c.items()
                 if kk not in ('textos','rotulos','_chaves','idx_treino','idx_val','idx_teste')}
             for k, c in CORPORA.items()},
    particoes={k: dict(n_treino=len(c['idx_treino']), n_val=len(c['idx_val']),
                       n_teste=len(c['idx_teste'])) for k, c in CORPORA.items()},
    resultados=RESULTADOS,
    agregado=AGG.to_dict(orient='records'),
    decomposicao=DEC.to_dict(orient='records'),
    regimes_limiar=REG.to_dict(orient='records'),
    retencao=RET.to_dict(orient='records'),
    comparacao_m3=(CMP.to_dict(orient='records') if CMP is not None else None),
    diagnosticos=DIAGNOSTICOS,
    saneamento_codificacao=SANEADOS,
    tempos_execucao=TEMPOS_MIN,
)

arq_json = os.path.join(OUT_DIR, f'M4_XLMR_resultados_{TAG}.json')
with open(arq_json, 'w', encoding='utf-8') as f:
    json.dump(SAIDA, f, ensure_ascii=False, indent=2, default=float)

arq_csv = os.path.join(OUT_DIR, f'M4_XLMR_matriz_{TAG}.csv')
AGG.to_csv(arq_csv, index=False)

print('Files saved:')
for a in (arq_json, arq_csv, arq_erros):
    print(f'  · {a}  ({os.path.getsize(a)/1024:.0f} KB)')
print(f'  · {PRED_DIR}/  (predicted probabilities — re-analysis without retraining)')

## 22. Run summary

In [ ]:
total = sum(TEMPOS_MIN.values())
print('='*66)
print('  RUN TIMES (min)')
print('='*66)
for k, v in TEMPOS_MIN.items():
    if not k.startswith('avaliacao_'):
        print(f'  {k:<44} {v:>8.2f}')
t_aval = sum(v for k, v in TEMPOS_MIN.items() if k.startswith('avaliacao_'))
print(f"  {'avaliacao_* (aggregated)':<44} {t_aval:>8.2f}")
print('-'*66)
print(f"  {'TOTAL':<44} {total:>8.2f} min  ({total/60:.2f} h)")

print('\n' + '='*66)
print('  SUMMARY')
print('='*66)
diag_m = AGG[AGG.treino==AGG.teste]['macro_f1_05_media'].mean()
grpA   = AGG[AGG.grupo=='A']['macro_f1_05_media'].mean()
grpB   = AGG[AGG.grupo=='B']['macro_f1_05_media'].mean()
grpC   = AGG[AGG.grupo=='C']['macro_f1_05_media'].mean()
print(f'  Diagonal (in-corpus) .................... {diag_m:.3f}')
print(f'  Group A (same language, other domain) ... {grpA:.3f}   ΔF1 = {diag_m-grpA:+.3f}')
print(f'  Group B (other language, close domain) .. {grpB:.3f}   ΔF1 = {diag_m-grpB:+.3f}')
print(f'  Group C (maximum distance) .............. {grpC:.3f}   ΔF1 = {diag_m-grpC:+.3f}')

abaixo = AGG[AGG.ganho_sobre_piso <= 0]
print(f'\n  Cells below the trivial floor of the target: {len(abaixo)} of {len(AGG)}')
if len(abaixo):
    print('  ' + ', '.join(f"{r['treino']}→{r['teste']}" for _, r in abaixo.iterrows()))
    print('  → In these cells the model is worse than predicting the majority class.')
    print('    This is a result in itself.')

if CMP is not None:
    print('\n── M4 minus M3 (regime R2) ──')
    for g, v in CMP.groupby('grupo')['d_R2'].mean().items():
        print(f'  {g:<10} {v:+.3f}')
    print(f"  {'OVERALL':<10} {CMP['d_R2'].mean():+.3f}")

print('\n  Full 6x6 matrix written.')
print('')